# `run_train_v3.py` Complete Local Dependency Notebook

This notebook collects the code used by `train/run_train_v3.py` and the local files it imports.

The code cells are ordered so the notebook can be read top-to-bottom as a standalone local-code copy of the training stack.

Included local modules:
- `train/run_train_v3.py`
- `data/loader/build_dataset.py`
- `data/loader/splits.py`
- `data/augmentation.py`
- `data/loader/ffpp_dataset_v2.py`
- `models/backbone.py`
- `models/branches/spatial_branch.py`
- `models/branches/frequency_branch_raw.py`
- `models/branches/noise_branch_raw.py`
- `models/fusion/cgaf.py`
- `models/temporal/temporal_model.py`
- `models/afag_net_v3.py`
- `train/psuedo_mask.py`
- `train/train_pipeline_v3.py`

Note:
- External packages such as `torch`, `timm`, `cv2`, `onnxruntime`, `insightface`, `decord`, and others are still external dependencies.

## Entry Script

Source: `train/run_train_v3.py`

In [ ]:
"""
run_train_v3.py  — Entry point for AFAGNet v3 training
=======================================================
Auto-detects GPU VRAM and selects hyperparameters accordingly.

BATCH SIZE SCALING TABLE
─────────────────────────────────────────────────────────────
                    batch=2         batch=4         batch=8
                    (RTX 3050 4GB)  (T4 safe)       (T4 full)
─────────────────────────────────────────────────────────────
weight_decay        5e-4            5e-4            3e-4
mixup_alpha         0.2             0.3             0.4
lr                  3e-4            3e-4            4e-4
warmup_epochs       2               2               3
patience            7               7               5
num_workers         0               2               4
focal_alpha         0.25            0.25            0.25  (FIXED from 0.5)
─────────────────────────────────────────────────────────────
Override: set FORCE_BATCH_SIZE to 2, 4, or 8 below.
"""

import os
import sys
import torch
import warnings

from torch.utils.data import Subset

warnings.filterwarnings("ignore", message=".*HF Hub.*",      category=UserWarning)
warnings.filterwarnings("ignore", message=".*rcond.*",        category=FutureWarning)

FORCE_BATCH_SIZE = None   # set to 2, 4, or 8 to override auto-detection

HPARAM_TABLE = {
    2: dict(weight_decay=5e-4, mixup_alpha=0.2, lr=3e-4,
            warmup_epochs=2, patience=7, num_workers=0, focal_alpha=0.25,
            note="RTX 3050 / 4GB GPU"),
    4: dict(weight_decay=5e-4, mixup_alpha=0.3, lr=3e-4,
            warmup_epochs=2, patience=7, num_workers=2, focal_alpha=0.25,
            note="T4 conservative"),
    8: dict(weight_decay=3e-4, mixup_alpha=0.4, lr=4e-4,
            warmup_epochs=3, patience=5, num_workers=4, focal_alpha=0.25,
            note="T4 full-performance (recommended)"),
}


def detect_batch_size():
    if not torch.cuda.is_available():
        return 2
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 14:
        return 8
    elif vram_gb >= 8:
        return 4
    return 2

def main():
    # GPU detection + hyperparameter selection
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}  {p.total_memory/(1024**3):.1f}GB")

    batch_size = FORCE_BATCH_SIZE if FORCE_BATCH_SIZE else detect_batch_size()
    if batch_size not in HPARAM_TABLE:
        batch_size = 2
    hp = HPARAM_TABLE[batch_size]
    print(f"\nBatch: {batch_size}  [{hp['note']}]  "
          f"lr={hp['lr']}  wd={hp['weight_decay']}  "
          f"mixup={hp['mixup_alpha']}  focal_α={hp['focal_alpha']}\n")

    root      = "FaceForensics_Data"
    cache_dir = "dataset/cache/ffpp_processed_v2"

    video_paths, labels, mask_paths, domains = build_ffpp_dataset(root)

    # ----------------------------------------------------------------
    # STEP 1: Build v2 cache using FFPPDatasetV2 directly
    # (The old build_cache.py uses FFPPDataset v1 — wrong class, wrong
    #  cache signature. Must use FFPPDatasetV2 to get the right .pt files.)
    # ----------------------------------------------------------------
    cache_builder = FFPPDatasetV2(
        video_paths=video_paths,
        labels=labels,
        mask_paths=mask_paths,
        domains=domains,
        cache_dir=cache_dir,
        use_alignment=True,
        training_mode=False,    # never augment during cache build
        label_smoothing=0.0,
        use_sbi=False,
        temporal_strategy="blend",
    )

    stats = cache_builder.get_cache_stats()
    print(f"Cache directory : {cache_dir}")
    print(f"Cache status    : {stats['cached']}/{stats['total']} cached, {stats['missing']} missing")

    if stats["missing"] > 0:
        print(f"\nBuilding {stats['missing']} missing cache entries (face detection + alignment).")
        print("This only runs once. Grab a coffee — ~3-4 hours for 9431 videos.")
        print("You can interrupt and restart; completed entries are not rebuilt.\n")
        cache_builder.init_face_detector()
        cache_builder.build_cache(num_workers=0)   # 0 workers = safe on Windows
        stats = cache_builder.get_cache_stats()
        print(f"\nCache complete: {stats['cached']}/{stats['total']}")

    # Guard: refuse to train if fewer than 80% of samples are cached
    # (training on live-detected frames is 50-100× slower per batch)
    cache_pct = stats["cached"] / max(stats["total"], 1)
    if cache_pct < 0.80:
        print(f"\nERROR: Only {stats['cached']}/{stats['total']} ({cache_pct*100:.1f}%) samples cached.")
        print("Training on uncached samples is extremely slow (5+ s/it vs 0.3 s/it).")
        print("Run this script again — cache building will resume where it left off.")
        sys.exit(1)

    # ----------------------------------------------------------------
    # STEP 2: Create train / val datasets with correct flags
    # ----------------------------------------------------------------
    train_indices, val_indices = build_identity_disjoint_split(
        video_paths, val_ratio=0.2, seed=42
    )

    # Training dataset: augmentation ON, label smoothing ON
    train_dataset = Subset(
        FFPPDatasetV2(
            video_paths=video_paths, labels=labels,
            mask_paths=mask_paths, domains=domains,
            cache_dir=cache_dir,
            use_alignment=True,
            training_mode=True,      # ← augmentation ON
            label_smoothing=0.05,
            use_sbi=True,
            temporal_strategy="blend",
        ),
        train_indices,
    )

    # Validation dataset: augmentation OFF, no smoothing
    val_dataset = Subset(
        FFPPDatasetV2(
            video_paths=video_paths, labels=labels,
            mask_paths=mask_paths, domains=domains,
            cache_dir=cache_dir,
            use_alignment=True,
            training_mode=False,     # ← augmentation OFF
            label_smoothing=0.0,
            use_sbi=False,
            temporal_strategy="blend",
        ),
        val_indices,
    )

    print(f"\nIdentity-aware split: train={len(train_dataset)}, val={len(val_dataset)}")

    # ----------------------------------------------------------------
    # STEP 3: Model — load best checkpoint if it exists (resume)
    # ----------------------------------------------------------------
    # Set your target total epochs here. If resuming, training continues
    # from where it left off up to this number.
    TOTAL_EPOCHS = 30   # ← change this to train longer

    model = AFAGNetV3(use_gradient_checkpointing=False)
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"AFAGNetV3 params: {total:,} total / {trainable:,} trainable")

    checkpoint_path = "checkpoints/best_model.pth"
    resume_epoch    = 0
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
        log_path = "experiments/logs/epoch_results_v3.txt"
        if os.path.exists(log_path):
            with open(log_path) as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]
            if lines:
                try:
                    resume_epoch = int(lines[-1].split("epoch=")[1].split(",")[0])
                except Exception:
                    resume_epoch = 0

        if resume_epoch >= TOTAL_EPOCHS:
            print(f"\nAlready completed {resume_epoch} epochs (target={TOTAL_EPOCHS}).")
            print(f"To train more, increase TOTAL_EPOCHS above {resume_epoch} in run_train_v3.py")
            return
        print(f"\nResuming from epoch {resume_epoch} → training epochs {resume_epoch+1}–{TOTAL_EPOCHS}")
    else:
        print("\nNo checkpoint found — training from scratch.")

    # ----------------------------------------------------------------
    # STEP 4: Train
    # ----------------------------------------------------------------
    history = train(
        model=model,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        epochs=TOTAL_EPOCHS,
        batch_size=batch_size,
        num_workers=hp["num_workers"],
        lr=hp["lr"],
        backbone_lr_factor=0.1,
        weight_decay=hp["weight_decay"],
        patience=hp["patience"],
        warmup_epochs=hp["warmup_epochs"],
        use_amp=True,
        focal_alpha=hp["focal_alpha"],   # 0.25 — upweights real class
        focal_gamma=2.0,
        use_mixup=True,
        mixup_alpha=hp["mixup_alpha"],
        use_ema=True,
        ema_decay=0.9999,
        use_alt_freezing=False,
        save_dir="checkpoints",
        log_dir="experiments/logs",
        resume_epoch=resume_epoch,
    )

    print(f"\nBest Val AUC: {max(history.get('val_auc', [0])):.4f}")


if __name__ == "__main__":
    main()


## Dataset Builder

Source: `data/loader/build_dataset.py`

In [ ]:
import os


def build_ffpp_dataset(root_dir, compression="c23"):
    video_paths = []
    labels = []
    mask_paths = []
    domains = []

    manipulated_root = os.path.join(root_dir, "manipulated_sequences")
    original_root = os.path.join(root_dir, "original_sequences")

    # -------------------------
    # REAL VIDEOS (YouTube + Actors)
    # -------------------------
    youtube_dir = os.path.join(original_root, "youtube")
    actors_dir = os.path.join(original_root, "actors")

    # YouTube
    for root, _, files in os.walk(youtube_dir):
        for f in files:
            if f.endswith(".mp4"):
                video_paths.append(os.path.join(root, f))
                labels.append(0)
                mask_paths.append(None)
                domains.append("youtube")

    # Actors
    for root, _, files in os.walk(actors_dir):
        for f in files:
            if f.endswith(".mp4"):
                video_paths.append(os.path.join(root, f))
                labels.append(0)
                mask_paths.append(None)
                domains.append("actors")

    # -------------------------
    # FAKE VIDEOS
    # -------------------------
    fake_types = os.listdir(manipulated_root)

    for fake_type in fake_types:
        fake_path = os.path.join(manipulated_root, fake_type)

        video_dir = os.path.join(fake_path, compression)
        mask_dir = os.path.join(fake_path, "masks")

        for root, _, files in os.walk(video_dir):
            for f in files:
                if f.endswith(".mp4"):
                    video_paths.append(os.path.join(root, f))
                    labels.append(1)
                    domains.append(fake_type)

                    # FaceShifter has no masks
                    if fake_type == "FaceShifter":
                        mask_paths.append(None)
                    else:
                        # Mask video path
                        mask_path = os.path.join(mask_dir, "videos", f)

                        if os.path.exists(mask_path):
                            mask_paths.append(mask_path)
                        else:
                            mask_paths.append(None)

    return video_paths, labels, mask_paths, domains


## Dataset Splitter

Source: `data/loader/splits.py`

In [ ]:
from pathlib import Path

import torch


def extract_identity_tokens(video_path):
    stem = Path(video_path).stem
    prefix = stem.split("__", 1)[0]
    numeric_tokens = [token for token in prefix.split("_") if token.isdigit()]
    if numeric_tokens:
        return tuple(sorted(set(numeric_tokens)))
    return (stem,)


def build_identity_disjoint_split(video_paths, val_ratio=0.2, seed=42):
    identity_to_indices = {}

    for idx, video_path in enumerate(video_paths):
        for token in extract_identity_tokens(video_path):
            identity_to_indices.setdefault(token, []).append(idx)

    identity_keys = sorted(identity_to_indices)
    if not identity_keys:
        raise ValueError("No identity groups were found for dataset splitting.")

    generator = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(identity_keys), generator=generator).tolist()
    target_val_identities = max(1, int(round(len(identity_keys) * val_ratio)))

    val_indices = set()
    for perm_idx in perm[:target_val_identities]:
        val_indices.update(identity_to_indices[identity_keys[perm_idx]])

    all_indices = set(range(len(video_paths)))
    train_indices = sorted(all_indices - val_indices)
    val_indices = sorted(val_indices)
    return train_indices, val_indices


## Augmentation

Source: `data/augmentation.py`

In [ ]:
"""
augmentation.py  — Video-consistent augmentation for deepfake detection training
=================================================================================
Key design principle:
    ALL augmentations make ONE random decision and apply it IDENTICALLY to
    every frame in the video sequence. Applying different transforms per frame
    would create artificial temporal inconsistencies that bias the temporal model.

Techniques included:
    1. JPEG compression simulation — critical for FF++ generalization
       (FF++ C23 uses heavy compression; models must handle varied quality)
    2. Random Gaussian noise — robustness augmentation
    3. Color jitter — subtle face-level variation
    4. Horizontal flip (video-consistent)
    5. Random erasing — occlusion robustness
    6. Frequency-domain noise — exposes models to frequency artifacts during training
    
Usage:
    # In your training loop or dataset __getitem__:
    augmenter = VideoAugmentation(training=True)
    frames_tensor = augmenter(frames_tensor)  # (N, C, H, W)
    
    # frames_tensor should be in [-1, 1] range (your rgb_ycbcr normalization)
    
Research basis:
    SBI (CVPR 2022): self-blended augmentation for generalization
    LipForensics (CVPR 2021): augmentation for temporal deepfake detection
    LSDA (ICCV 2023): large-scale augmentation strategy
"""

import io
import random
from typing import Optional, Tuple

import torch
import torch.nn.functional as F
import numpy as np

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False


class VideoAugmentation:
    """
    Video-level augmentation: ONE random decision per video, applied to ALL frames.
    
    Input/Output tensor shape: (N, C, H, W) where
        N = number of frames (e.g. 16)
        C = channels (6 for RGB+YCbCr, or 3)
        H, W = 224, 224
        Value range: [-1, 1]  (your rgb_ycbcr normalization)
    """

    def __init__(
        self,
        training: bool = True,
        # Augmentation probabilities
        flip_prob: float = 0.5,
        jpeg_prob: float = 0.4,
        noise_prob: float = 0.3,
        color_jitter_prob: float = 0.3,
        erase_prob: float = 0.1,
        freq_noise_prob: float = 0.2,
        # Augmentation parameters
        jpeg_quality_range: Tuple[int, int] = (55, 95),
        noise_std_range: Tuple[float, float] = (0.005, 0.03),
        brightness_range: Tuple[float, float] = (-0.1, 0.1),
        contrast_range: Tuple[float, float] = (0.9, 1.1),
        saturation_range: Tuple[float, float] = (0.95, 1.05),
        erase_scale_range: Tuple[float, float] = (0.02, 0.12),
        # Channel mode: 'rgb' for 3ch, 'rgb_ycbcr' for 6ch
        channel_mode: str = "rgb_ycbcr",
    ):
        self.training         = training
        self.flip_prob        = flip_prob
        self.jpeg_prob        = jpeg_prob
        self.noise_prob       = noise_prob
        self.color_jitter_prob = color_jitter_prob
        self.erase_prob       = erase_prob
        self.freq_noise_prob  = freq_noise_prob

        self.jpeg_quality_range  = jpeg_quality_range
        self.noise_std_range     = noise_std_range
        self.brightness_range    = brightness_range
        self.contrast_range      = contrast_range
        self.saturation_range    = saturation_range
        self.erase_scale_range   = erase_scale_range
        self.channel_mode        = channel_mode

    # -----------------------------------------------------------------------
    # Individual transforms
    # -----------------------------------------------------------------------

    def horizontal_flip(self, x: torch.Tensor) -> torch.Tensor:
        """Flip all frames horizontally. x: (N, C, H, W)"""
        return torch.flip(x, dims=[-1])

    def jpeg_simulate(self, x: torch.Tensor, quality: int) -> torch.Tensor:
        """
        Simulate JPEG compression artifacts.
        
        Critical for FF++ generalization:
            FF++ C23 = quality 23 compression during dataset creation
            Training without JPEG aug → model learns non-compressed artifacts only
            → Fails on social media / real-world videos which are re-compressed
        
        x: (N, C, H, W) in [-1, 1], operates only on first 3 (RGB) channels
        """
        if not PIL_AVAILABLE:
            return x  # skip if PIL not available

        N, C, H, W = x.shape
        # Only apply to RGB channels (first 3) to avoid corrupting YCbCr
        rgb_channels = min(3, C)

        # Convert from [-1, 1] → [0, 255]
        x_out = x.clone()
        for n in range(N):
            try:
                frame_rgb = x[n, :rgb_channels]
                frame_uint8 = ((frame_rgb + 1.0) / 2.0 * 255.0).clamp(0, 255)
                frame_uint8 = frame_uint8.byte().permute(1, 2, 0).cpu().numpy()

                pil_img = Image.fromarray(frame_uint8)
                buffer  = io.BytesIO()
                pil_img.save(buffer, format="JPEG", quality=quality)
                buffer.seek(0)
                pil_compressed = Image.open(buffer).convert("RGB")

                compressed = torch.tensor(
                    np.array(pil_compressed), dtype=x.dtype, device=x.device
                ).permute(2, 0, 1)   # (3, H, W)

                # Back to [-1, 1]
                x_out[n, :rgb_channels] = (compressed / 255.0) * 2.0 - 1.0
            except Exception:
                pass  # If JPEG fails for any frame, skip it

        return x_out

    def gaussian_noise(self, x: torch.Tensor, std: float) -> torch.Tensor:
        """Add Gaussian noise, same std across frames, different noise realization."""
        noise = torch.randn_like(x) * std
        return torch.clamp(x + noise, -1.0, 1.0)

    def color_jitter(
        self,
        x: torch.Tensor,
        brightness: float,
        contrast: float,
        saturation: float,
    ) -> torch.Tensor:
        """
        Subtle face-level color augmentation.
        Uses same parameters for all frames to maintain temporal consistency.
        Only applied to RGB channels (first 3).
        """
        x_out = x.clone()
        rgb_channels = min(3, x.shape[1])

        for n in range(x.shape[0]):
            frame = x_out[n, :rgb_channels]  # (3, H, W) in [-1, 1]

            # Brightness: add constant
            frame = frame + brightness

            # Contrast: scale around mean
            mean = frame.mean(dim=[1, 2], keepdim=True)
            frame = mean + contrast * (frame - mean)

            # Saturation: interpolate between frame and grayscale
            # (Only meaningful for RGB channels)
            if rgb_channels == 3:
                gray = 0.299 * frame[0] + 0.587 * frame[1] + 0.114 * frame[2]
                gray = gray.unsqueeze(0).expand_as(frame)
                frame = saturation * frame + (1.0 - saturation) * gray

            x_out[n, :rgb_channels] = torch.clamp(frame, -1.0, 1.0)

        return x_out

    def random_erase(
        self,
        x: torch.Tensor,
        scale: float,
        fill_value: float = 0.0,
    ) -> torch.Tensor:
        """
        Erase a random rectangular region — same region for all frames.
        Teaches the model to detect fakes from partial face views.
        """
        N, C, H, W = x.shape
        area    = H * W * scale
        aspect  = random.uniform(0.5, 2.0)
        h_erase = int((area / aspect) ** 0.5)
        w_erase = int((area * aspect) ** 0.5)

        h_erase = min(h_erase, H)
        w_erase = min(w_erase, W)

        top  = random.randint(0, H - h_erase)
        left = random.randint(0, W - w_erase)

        x_out = x.clone()
        x_out[:, :, top:top + h_erase, left:left + w_erase] = fill_value
        return x_out

    def frequency_domain_noise(self, x: torch.Tensor, strength: float = 0.05) -> torch.Tensor:
        """
        Add noise in the frequency domain — simulates GAN artifacts.
        
        Real GAN images produce specific frequency patterns (spectral artifacts).
        By training with frequency noise, the model learns to generalize across
        different GAN architectures' artifact signatures.
        
        Works on float32 internally; casts back to original dtype.
        """
        orig_dtype = x.dtype
        x_f32 = x.float()  # Always float32 for FFT

        N, C, H, W = x_f32.shape

        # FFT per frame
        fft = torch.fft.rfft2(x_f32, norm='ortho')

        # Add small random perturbation to high-frequency components
        # (low-frequency is left clean to preserve face structure)
        noise_mag = torch.randn_like(fft.real) * strength
        noise_phase = torch.randn_like(fft.real) * strength * 0.5

        # Apply only to high-frequency half of spectrum
        H_mid = fft.shape[-2] // 2
        W_mid = fft.shape[-1] // 2
        fft.real[:, :, H_mid:, W_mid:] += noise_mag[:, :, H_mid:, W_mid:]
        fft.imag[:, :, H_mid:, W_mid:] += noise_phase[:, :, H_mid:, W_mid:]

        # Inverse FFT
        x_perturbed = torch.fft.irfft2(fft, s=(H, W), norm='ortho')
        x_perturbed = torch.clamp(x_perturbed, -1.0, 1.0)

        return x_perturbed.to(orig_dtype)

    # -----------------------------------------------------------------------
    # Main call
    # -----------------------------------------------------------------------

    def __call__(self, frames_tensor: torch.Tensor) -> torch.Tensor:
        """
        Apply video-consistent augmentation.
        
        frames_tensor: (N, C, H, W), values in [-1, 1]
        returns: augmented tensor, same shape and dtype
        """
        if not self.training:
            return frames_tensor

        x = frames_tensor

        # --- 1. Horizontal flip (single random decision for all frames) ---
        if random.random() < self.flip_prob:
            x = self.horizontal_flip(x)

        # --- 2. JPEG compression simulation ---
        if random.random() < self.jpeg_prob:
            quality = random.randint(*self.jpeg_quality_range)
            x = self.jpeg_simulate(x, quality)

        # --- 3. Color jitter ---
        if random.random() < self.color_jitter_prob:
            brightness = random.uniform(*self.brightness_range)
            contrast   = random.uniform(*self.contrast_range)
            saturation = random.uniform(*self.saturation_range)
            x = self.color_jitter(x, brightness, contrast, saturation)

        # --- 4. Gaussian noise ---
        if random.random() < self.noise_prob:
            std = random.uniform(*self.noise_std_range)
            x = self.gaussian_noise(x, std)

        # --- 5. Random erasing ---
        if random.random() < self.erase_prob:
            scale = random.uniform(*self.erase_scale_range)
            x = self.random_erase(x, scale)

        # --- 6. Frequency domain noise ---
        if random.random() < self.freq_noise_prob:
            strength = random.uniform(0.01, 0.05)
            x = self.frequency_domain_noise(x, strength)

        return x


# -----------------------------------------------------------------------
# Integration helper: wrap your dataset __getitem__ with this
# -----------------------------------------------------------------------
def augment_sample(sample: dict, augmenter: VideoAugmentation) -> dict:
    """
    Applies augmentation to a cached dataset sample dict.
    
    Expected keys: "frames" (N, C, H, W), "mask_frames" (N, H, W), etc.
    The mask is NOT augmented (only flipped if frames are flipped — handle separately).
    
    Usage:
        augmenter = VideoAugmentation(training=True)
        
        # In your training dataset's __getitem__:
        sample = self.load_from_cache(idx)
        sample = augment_sample(sample, augmenter)
    """
    augmented = dict(sample)
    frames = sample["frames"]  # (N, C, H, W)

    if augmenter.training:
        # Decide flip BEFORE augmentation
        do_flip = random.random() < augmenter.flip_prob

        if do_flip:
            frames = torch.flip(frames, dims=[-1])
            # Also flip masks to maintain spatial correspondence
            if "mask_frames" in sample and sample["mask_frames"] is not None:
                augmented["mask_frames"] = torch.flip(sample["mask_frames"], dims=[-1])

        # Apply remaining augmentations (flip was already applied or skipped)
        original_flip_prob = augmenter.flip_prob
        augmenter.flip_prob = 0.0  # disable flip in main call since we handled it
        frames = augmenter(frames)
        augmenter.flip_prob = original_flip_prob

    augmented["frames"] = frames
    return augmented


# -----------------------------------------------------------------------
# Self-Blended Images (SBI) augmentation — highest-impact for generalization
# Reference: "Detecting Deepfakes with Self-Blended Images" (CVPR 2022)
# -----------------------------------------------------------------------
class SBIAugmentation:
    """
    Creates synthetic fake training samples by blending regions of REAL faces.
    
    Algorithm:
    1. Take a real face frame
    2. Warp/transform a copy of it slightly (affine, elastic deformation)
    3. Create a binary or smooth blending mask
    4. Blend: fake_frame = mask * warped + (1-mask) * original
    5. Label the result as FAKE (1) with the blending mask as ground truth
    
    This teaches the model to detect blending boundaries without needing
    a diverse collection of fake generation methods.
    
    Simple version provided here — for production use, see the original paper.
    """

    def __init__(
        self,
        prob: float = 0.3,
        alpha_range: Tuple[float, float] = (0.3, 0.7),
    ):
        self.prob        = prob
        self.alpha_range = alpha_range

    def _simple_warp(self, x: torch.Tensor) -> torch.Tensor:
        """
        Simple affine warp — approximate face landmark displacement.
        x: (C, H, W) single frame in [-1, 1]
        """
        C, H, W = x.shape
        # Small random affine transform
        angle = random.uniform(-5, 5) * (torch.pi / 180)
        scale = random.uniform(0.95, 1.05)
        tx    = random.uniform(-5, 5) / W
        ty    = random.uniform(-5, 5) / H

        cos_a = torch.tensor([[scale * torch.cos(angle), -scale * torch.sin(angle), tx],
                              [scale * torch.sin(angle),  scale * torch.cos(angle), ty]])

        grid = F.affine_grid(cos_a.unsqueeze(0), (1, C, H, W), align_corners=False)
        return F.grid_sample(x.unsqueeze(0), grid, align_corners=False, padding_mode='border').squeeze(0)

    def _gaussian_mask(self, H: int, W: int, device) -> torch.Tensor:
        """
        Generate a smooth elliptical blending mask centered in the face region.
        """
        cy, cx = H // 2 + random.randint(-H//8, H//8), W // 2 + random.randint(-W//8, W//8)
        ry     = H // 4 + random.randint(-H//8, H//8)
        rx     = W // 4 + random.randint(-W//8, W//8)

        yy, xx = torch.meshgrid(torch.arange(H, device=device),
                                torch.arange(W, device=device), indexing='ij')
        mask = ((yy - cy) ** 2 / (ry ** 2 + 1e-6) + (xx - cx) ** 2 / (rx ** 2 + 1e-6))
        mask = torch.exp(-mask)
        # Soft threshold
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-6)
        return mask.unsqueeze(0)  # (1, H, W)

    def __call__(self, sample: dict) -> dict:
        """
        With probability self.prob, convert a REAL sample to a SBI fake sample.
        
        Only applies to REAL samples (label=0).
        Returns modified sample with label=1 and pseudo blending mask.
        """
        if not torch.is_tensor(sample["label"]):
            label = float(sample["label"])
        else:
            label = sample["label"].item()

        if label != 0 or random.random() > self.prob:
            return sample

        frames = sample["frames"]   # (N, C, H, W)
        N, C, H, W = frames.shape

        # Blend all frames with the SAME mask but DIFFERENT warp realizations
        alpha     = random.uniform(*self.alpha_range)
        blend_mask = self._gaussian_mask(H, W, frames.device)  # (1, H, W)

        blended_frames = []
        for n in range(N):
            frame       = frames[n]         # (C, H, W)
            warped      = self._simple_warp(frame)
            blended     = alpha * blend_mask * warped + (1 - alpha * blend_mask) * frame
            blended_frames.append(blended.clamp(-1.0, 1.0))

        new_sample = dict(sample)
        new_sample["frames"]      = torch.stack(blended_frames)
        new_sample["label"]       = torch.tensor(1.0, dtype=sample["label"].dtype if torch.is_tensor(sample["label"]) else torch.float32)
        new_sample["mask_frames"] = blend_mask.squeeze(0).expand(N, H, W)
        new_sample["has_mask"]    = True

        return new_sample


## Dataset Class

Source: `data/loader/ffpp_dataset_v2.py`

In [ ]:
"""
ffpp_dataset_v2.py  — Research-level FF++ Dataset with Face Alignment + Augmentation
======================================================================================
Key improvements over v1:

  FIX-1: Face ALIGNMENT added after detection
          SOTA deepfake detectors (SBI, RECCE, AltFreezing) all align faces using
          5-point landmarks BEFORE cropping. This removes pose variation and lets
          the network focus on manipulation artifacts instead of pose differences.
          Alignment makes blending boundaries consistent across frames.

  FIX-2: Augmentation properly hooked into __getitem__
          v1 had VideoAugmentation defined but NEVER called it during training.
          This is a major reason for poor generalization on compressed videos.
          Augmentation is ONLY applied at training time (not cached) via a flag.

  FIX-3: Fast sampler weight extraction
          v1's make_balanced_sampler() iterated through every sample to read labels.
          With 6000+ samples this takes ~10 minutes. v2 reads labels directly from
          the dataset's label list — O(N) without any disk access.

  FIX-4: Clamped YCbCr normalization
          v1's rgb_ycbcr: output was not clamped, so extreme pixel values (e.g.
          from augmentation) could produce values outside [-1, 1], causing NaN
          in downstream batch normalization layers.

  NEW-1: Label smoothing support
          Returns smooth_label in the sample dict if label_smoothing > 0.
          Helps prevent overconfident predictions → better calibration.

  NEW-2: Temporal stride selection (uniform + motion-weighted blend)
          Instead of pure motion-based selection (which can miss important
          static frames), v2 blends motion scores with uniform sampling pressure.
          This prevents clustering all selected frames in one scene transition.

Research basis:
  - AltFreezing (ECCV 2023): face alignment preprocessing
  - SBI (CVPR 2022): augmentation during training on cached data
  - DeepfakeBench: standard protocol — 32 frames, aligned crops, c23
"""

import os
import warnings
import hashlib
import threading
import cv2
import torch
import numpy as np
import onnxruntime as ort
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

import insightface
from insightface.app import FaceAnalysis

from facenet_pytorch import MTCNN
from decord import VideoReader, cpu

# ---- augmentation import (safe — fails gracefully if file not on path) ----
try:
    _AUG_AVAILABLE = True
except ImportError:
    _AUG_AVAILABLE = False

np.seterr(all='ignore')
warnings.filterwarnings("ignore", message=".*rcond parameter will change.*", category=FutureWarning)


# ===========================================================================
# Worker globals (unchanged — safe for multiprocessing)
# ===========================================================================
_WORKER_DATASET  = None
_WORKER_DETECTOR = None

def _build_cache_index_worker(dataset_kwargs, idx):
    global _WORKER_DETECTOR, _WORKER_DATASET
    if _WORKER_DATASET is None:
        _WORKER_DATASET = FFPPDatasetV2(**dataset_kwargs)
    if _WORKER_DETECTOR is None:
        _WORKER_DATASET.init_face_detector()
        _WORKER_DETECTOR = _WORKER_DATASET.face_detector
    cache_path = _WORKER_DATASET.get_cache_path(idx)
    sample = _WORKER_DATASET.build_sample(idx)
    _WORKER_DATASET.save_cached_sample(cache_path, sample)
    return idx


# ===========================================================================
# FACE ALIGNMENT UTILITY
# ===========================================================================

# Reference 5 facial landmarks for a canonical 112x112 aligned face
# (ArcFace standard, used by InsightFace, AltFreezing, SBI, etc.)
REFERENCE_LANDMARKS_112 = np.array([
    [38.2946, 51.6963],
    [73.5318, 51.5014],
    [56.0252, 71.7366],
    [41.5493, 92.3655],
    [70.7299, 92.2041],
], dtype=np.float32)

# Scale to 224x224
REFERENCE_LANDMARKS_224 = REFERENCE_LANDMARKS_112 * (224 / 112)


def align_face_5pt(img: np.ndarray, landmarks_5pt: np.ndarray, output_size: int = 224) -> np.ndarray:
    """
    Align a face image using 5 facial landmarks.
    Uses similarity transform (rotation + scale + translation only).

    img:           (H, W, 3) BGR or RGB numpy array
    landmarks_5pt: (5, 2) float32 — predicted [leye, reye, nose, lmouth, rmouth]
    output_size:   output image size (default 224)

    Returns: (output_size, output_size, 3) aligned face
    """
    ref = REFERENCE_LANDMARKS_224 * (output_size / 224)
    src = landmarks_5pt.astype(np.float32)

    tform = cv2.estimateAffinePartial2D(src, ref, method=cv2.LMEDS)[0]
    if tform is None:
        # Fallback: center crop if alignment fails
        h, w = img.shape[:2]
        side  = min(h, w)
        top   = (h - side) // 2
        left  = (w - side) // 2
        cropped = img[top:top+side, left:left+side]
        return cv2.resize(cropped, (output_size, output_size))

    aligned = cv2.warpAffine(img, tform, (output_size, output_size),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REPLICATE)
    return aligned


# ===========================================================================
# MAIN DATASET CLASS
# ===========================================================================

class FFPPDatasetV2(Dataset):
    """
    Improved FF++ dataset with face alignment and augmentation.

    New parameters vs v1:
        use_alignment:     bool  — align face using 5-pt landmarks (default True)
        training_mode:     bool  — enables augmentation in __getitem__
        label_smoothing:   float — smoothing factor for labels (default 0.05)
        use_sbi:           bool  — apply Self-Blended Images augmentation
        temporal_strategy: str  — 'motion' | 'uniform' | 'blend' (default 'blend')
    """

    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __init__(
        self,
        video_paths,
        labels,
        mask_paths,
        domains,
        num_frames: int = 32,
        robust_mode=None,
        use_retinaface: bool = True,
        cache_dir=None,
        cache_version: str = "v2",        # bump from v1 → forces full rebuild
        use_alignment: bool = True,        # NEW: face alignment
        training_mode: bool = False,       # NEW: enables augmentation at getitem
        label_smoothing: float = 0.05,     # NEW: smooth labels slightly
        use_sbi: bool = True,              # NEW: Self-Blended Images augmentation
        temporal_strategy: str = "blend",  # NEW: frame selection strategy
    ):
        self.video_paths       = video_paths
        self.labels            = labels
        self.mask_paths        = mask_paths
        self.domains           = domains
        self.num_frames        = num_frames
        self.robust_mode       = robust_mode
        self.use_retinaface    = use_retinaface
        self.cache_dir         = Path(cache_dir) if cache_dir is not None else None
        self.cache_version     = cache_version
        self.use_alignment     = use_alignment
        self.training_mode     = training_mode
        self.label_smoothing   = label_smoothing
        self.use_sbi           = use_sbi
        self.temporal_strategy = temporal_strategy
        self.gpu_id            = 0

        self.device      = 'cuda' if torch.cuda.is_available() else 'cpu'
        if self.cache_dir is not None:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

        self.face_detector = None

        # Augmentation modules (lazy-initialized when training_mode=True)
        self._video_aug = None
        self._sbi_aug   = None

        # Base transform (resize + toTensor — alignment handles geometry)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    # -----------------------------------------------------------------------
    # AUGMENTATION LAZY INIT
    # -----------------------------------------------------------------------
    def _get_augmenters(self):
        if self._video_aug is None and _AUG_AVAILABLE:
            self._video_aug = VideoAugmentation(training=True)
        if self._sbi_aug is None and _AUG_AVAILABLE and self.use_sbi:
            self._sbi_aug = SBIAugmentation(prob=0.25)
        return self._video_aug, self._sbi_aug

    # -----------------------------------------------------------------------
    # FACE DETECTOR
    # -----------------------------------------------------------------------
    def init_face_detector(self):
        if self.face_detector is not None:
            return
        if self.use_retinaface:
            providers, ctx_id = self.get_insightface_runtime()
            self.face_detector = FaceAnalysis(name="buffalo_l", providers=providers)
            self.face_detector.prepare(ctx_id=self.gpu_id, det_size=(640, 640))

    def get_insightface_runtime(self):
        if not torch.cuda.is_available():
            return ["CPUExecutionProvider"], -1
        if hasattr(ort, "preload_dlls"):
            try:
                ort.preload_dlls()
            except Exception:
                pass
        try:
            providers = ort.get_available_providers()
        except Exception:
            providers = []
        if "CUDAExecutionProvider" in providers:
            return ["CUDAExecutionProvider", "CPUExecutionProvider"], 0
        return ["CPUExecutionProvider"], -1

    # -----------------------------------------------------------------------
    # DATASET PROTOCOL
    # -----------------------------------------------------------------------
    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        # Load from cache
        cache_path = self.get_cache_path(idx)
        if cache_path is not None and cache_path.exists():
            sample = torch.load(cache_path, map_location="cpu")
        else:
            sample = self.build_sample(idx)
            if cache_path is not None:
                self.save_cached_sample(cache_path, sample)

        # Apply augmentations at training time (NOT during cache build)
        if self.training_mode and _AUG_AVAILABLE:
            video_aug, sbi_aug = self._get_augmenters()

            # SBI augmentation: convert real → synthetic fake with probability
            if sbi_aug is not None and sample["label"].item() == 0:
                sample = sbi_aug(sample)

            # Video-consistent augmentation (flip, JPEG, noise, etc.)
            if video_aug is not None:
                sample = augment_sample(sample, video_aug)

        # FIX-4: Apply label smoothing for training
        if self.training_mode and self.label_smoothing > 0:
            lbl = sample["label"].item()
            eps = self.label_smoothing
            smooth = lbl * (1.0 - eps) + (1.0 - lbl) * eps
            sample = dict(sample)
            sample["label"] = torch.tensor(smooth, dtype=torch.float32)

        return sample

    # -----------------------------------------------------------------------
    # CACHE
    # -----------------------------------------------------------------------
    def cache_signature(self, idx):
        parts = [
            self.cache_version,
            self.video_paths[idx],
            str(self.mask_paths[idx]),
            str(self.labels[idx]),
            str(self.domains[idx]),
            str(self.num_frames),
            str(self.robust_mode),
            str(self.use_retinaface),
            str(self.use_alignment),      # NEW: alignment changes output
            str(self.temporal_strategy),  # NEW: strategy changes frame selection
        ]
        return hashlib.sha1("||".join(parts).encode("utf-8")).hexdigest()

    def get_cache_path(self, idx):
        if self.cache_dir is None:
            return None
        return self.cache_dir / f"{self.cache_signature(idx)}.pt"

    def save_cached_sample(self, cache_path, sample):
        temp_path = cache_path.with_suffix(".tmp")
        cpu_sample = {
            "frames":      sample["frames"].cpu(),
            "mask_frames": sample["mask_frames"].cpu(),
            "label":       sample["label"].cpu(),
            "has_mask":    bool(sample["has_mask"]),
            "domain":      sample["domain"],
        }
        torch.save(cpu_sample, temp_path)
        os.replace(temp_path, cache_path)

    def get_cache_stats(self):
        if self.cache_dir is None:
            return {"total": len(self), "cached": 0, "missing": len(self), "cache_dir": None}
        cached = sum(1 for i in range(len(self)) if self.get_cache_path(i).exists())
        return {
            "total":     len(self),
            "cached":    cached,
            "missing":   len(self) - cached,
            "cache_dir": str(self.cache_dir),
        }

    def export_init_kwargs(self):
        return {
            "video_paths":       self.video_paths,
            "labels":            self.labels,
            "mask_paths":        self.mask_paths,
            "domains":           self.domains,
            "num_frames":        self.num_frames,
            "robust_mode":       self.robust_mode,
            "use_retinaface":    self.use_retinaface,
            "cache_dir":         str(self.cache_dir) if self.cache_dir is not None else None,
            "cache_version":     self.cache_version,
            "use_alignment":     self.use_alignment,
            "training_mode":     False,   # workers never augment during build
            "label_smoothing":   0.0,
            "use_sbi":           False,
            "temporal_strategy": self.temporal_strategy,
        }

    def build_cache(self, overwrite=False, indices=None, num_workers=0):
        self.init_face_detector()
        if self.cache_dir is None:
            raise ValueError("Cache directory not configured.")

        before  = self.get_cache_stats()
        indices = indices if indices is not None else list(range(len(self)))

        pending = [i for i in indices if not self.get_cache_path(i).exists() or overwrite]
        progress = tqdm(total=len(indices), desc="Building Cache V2", dynamic_ncols=True)
        progress.update(len(indices) - len(pending))

        built = 0
        skipped = len(indices) - len(pending)
        worker_count = max(int(num_workers or 0), 1)

        if worker_count <= 1:
            for idx in pending:
                sample = self.build_sample(idx)
                self.save_cached_sample(self.get_cache_path(idx), sample)
                built += 1
                progress.update(1)
                if built % 200 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()
        else:
            dataset_kwargs = self.export_init_kwargs()
            with ProcessPoolExecutor(max_workers=worker_count) as executor:
                futures = [executor.submit(_build_cache_index_worker, dataset_kwargs, i) for i in pending]
                for f in as_completed(futures):
                    f.result()
                    built += 1
                    progress.update(1)

        progress.close()
        return self.get_cache_stats()

    # -----------------------------------------------------------------------
    # BUILD SAMPLE
    # -----------------------------------------------------------------------
    def build_sample(self, idx):
        video_path = self.video_paths[idx]
        label      = self.labels[idx]
        domain     = self.domains[idx]
        mask_path  = self.mask_paths[idx]

        frames, frame_indices = self.load_video_frames(video_path)

        raw_masks = self.load_mask_frames(mask_path, frame_indices) if mask_path is not None \
                    else [None] * len(frames)

        batch_results = self.extract_face_batch_aligned(frames)

        faces, aligned_masks = [], []
        for i, result in enumerate(batch_results):
            if result is None:
                continue
            face_tensor, bbox = result
            faces.append(self.rgb_ycbcr(face_tensor))

            mask = raw_masks[i]
            if mask is not None:
                aligned_masks.append(self.crop_mask(mask, bbox))
            elif mask_path is not None:
                aligned_masks.append(torch.zeros((224, 224), dtype=torch.float32))

        # Fallback if detection fails entirely
        if len(faces) == 0:
            faces = [self.rgb_ycbcr(self.transform(Image.fromarray(f))) for f in frames]
            if mask_path is not None:
                aligned_masks = [
                    self.prepare_full_mask(m) if m is not None
                    else torch.zeros((224, 224), dtype=torch.float32)
                    for m in raw_masks
                ]

        # Pad/clip to num_frames
        if len(faces) < self.num_frames:
            faces         += [faces[-1]] * (self.num_frames - len(faces))
            aligned_masks += [aligned_masks[-1]] * (self.num_frames - len(aligned_masks)) \
                             if aligned_masks else []
        else:
            faces         = faces[:self.num_frames]
            aligned_masks = aligned_masks[:self.num_frames]

        faces_tensor = torch.stack(faces)

        if mask_path is not None and len(aligned_masks) > 0:
            mask_frames = torch.stack(aligned_masks)
            has_mask    = True
        else:
            mask_frames = torch.zeros((self.num_frames, 224, 224))
            has_mask    = False

        return {
            "frames":      faces_tensor,
            "mask_frames": mask_frames,
            "label":       torch.tensor(float(label), dtype=torch.float32),
            "has_mask":    has_mask,
            "domain":      domain,
        }

    # -----------------------------------------------------------------------
    # VIDEO LOADING
    # -----------------------------------------------------------------------
    def load_video_frames(self, path):
        try:
            vr          = VideoReader(path, ctx=cpu(0))
            total       = len(vr)
            if total <= 0:
                raise ValueError(f"Empty video: {path}")

            sample_count = min(total, self.num_frames * 4)
            indices      = np.linspace(0, total - 1, sample_count).astype(int)
            frames_array = vr.get_batch(indices).asnumpy()
            frames       = list(frames_array)

            if len(frames) == 0:
                raise ValueError(f"No frames decoded: {path}")

            selected = self._select_frames(frames, indices)
            return [frames[i] for i in selected], [int(indices[i]) for i in selected]

        except Exception as e:
            return self._load_opencv_fallback(path)

    def _select_frames(self, frames, all_indices):
        """
        NEW-2: Blended selection strategy.
        Pure motion can cluster all frames in one scene cut.
        Blend = 0.6 * motion_rank + 0.4 * uniform_pressure.
        """
        n = len(frames)
        if n <= self.num_frames:
            return list(range(n))

        motion_scores = self._motion_scores(frames)

        if self.temporal_strategy == "motion":
            selected = np.argsort(motion_scores)[-self.num_frames:]
            return sorted(selected)

        elif self.temporal_strategy == "uniform":
            return sorted(np.linspace(0, n - 1, self.num_frames).astype(int).tolist())

        else:  # 'blend' — default
            motion_rank  = np.argsort(np.argsort(motion_scores)).astype(float) / max(n - 1, 1)
            uniform_rank = np.linspace(0, 1, n)
            combined     = 0.6 * motion_rank + 0.4 * uniform_rank
            selected     = np.argsort(combined)[-self.num_frames:]
            return sorted(selected)

    def _motion_scores(self, frames):
        scores = [0.0]
        for i in range(1, len(frames)):
            prev = cv2.cvtColor(frames[i-1], cv2.COLOR_RGB2GRAY)
            curr = cv2.cvtColor(frames[i],   cv2.COLOR_RGB2GRAY)
            flow = cv2.calcOpticalFlowFarneback(
                prev, curr,
                None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            # Magnitude of flow vectors
            mag = np.sqrt(flow[...,0]**2 + flow[...,1]**2)
            scores.append(float(np.mean(mag)))
        return scores

    def _load_opencv_fallback(self, path):
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open: {path}")
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        sample_count = min(max(total, 1), self.num_frames * 4)
        indices = np.linspace(0, max(total - 1, 0), sample_count).astype(int)
        frames, valid = [], []
        for fi in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if ret:
                frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                valid.append(fi)
        cap.release()
        if not frames:
            raise ValueError(f"No frames: {path}")
        selected = self._select_frames(frames, valid)
        return [frames[i] for i in selected], [int(valid[i]) for i in selected]

    # -----------------------------------------------------------------------
    # FIX-1: FACE DETECTION + ALIGNMENT
    # -----------------------------------------------------------------------
    def extract_face_batch_aligned(self, frames):
        """
        Extract and ALIGN faces using 5-point landmarks.
        Returns list of (tensor, bbox) or None per frame.
        """
        results = []
        for frame in frames:
            if frame is None:
                results.append(None)
                continue

            try:
                faces = self.face_detector.get(frame)
            except Exception:
                results.append(None)
                continue

            if not faces:
                results.append(None)
                continue

            face = faces[0]
            x1, y1, x2, y2 = map(int, face.bbox)
            x1, y1, x2, y2 = self.expand_and_clip_bbox(
                x1, y1, x2, y2, frame.shape[1], frame.shape[0]
            )

            if self.use_alignment and hasattr(face, 'kps') and face.kps is not None:
                # 5-point landmark alignment
                try:
                    aligned = align_face_5pt(frame, face.kps, output_size=224)
                    face_tensor = torch.from_numpy(aligned).permute(2, 0, 1).float() / 255.0
                except Exception:
                    # fallback to bbox crop
                    face_img    = Image.fromarray(frame[y1:y2, x1:x2])
                    face_tensor = self.transform(face_img)
            else:
                face_img    = Image.fromarray(frame[y1:y2, x1:x2])
                face_tensor = self.transform(face_img)

            results.append((face_tensor, (x1, y1, x2, y2)))

        return results

    # -----------------------------------------------------------------------
    # MASK LOADING
    # -----------------------------------------------------------------------
    def load_mask_frames(self, path, indices):
        try:
            vr          = VideoReader(path, ctx=cpu(0))
            total       = len(vr)
            safe        = [min(int(i), total - 1) for i in indices]
            frames_arr  = vr.get_batch(safe).asnumpy()
            return [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in frames_arr]
        except Exception:
            return [None] * len(indices)

    def expand_and_clip_bbox(self, x1, y1, x2, y2, width, height):
        w = x2 - x1
        h = y2 - y1
        x1 = max(int(x1 - 0.15 * w), 0)
        y1 = max(int(y1 - 0.15 * h), 0)
        x2 = min(int(x2 + 0.15 * w), width)
        y2 = min(int(y2 + 0.15 * h), height)
        return x1, y1, x2, y2

    def crop_mask(self, mask, bbox):
        x1, y1, x2, y2 = bbox
        mask = mask[y1:y2, x1:x2]
        if mask.size == 0:
            return torch.zeros((224, 224), dtype=torch.float32)
        mask = cv2.resize(mask, (224, 224)) / 255.0
        return torch.tensor(mask, dtype=torch.float32)

    def prepare_full_mask(self, mask):
        mask = cv2.resize(mask, (224, 224)) / 255.0
        return torch.tensor(mask, dtype=torch.float32)

    # -----------------------------------------------------------------------
    # RGB + YCbCr NORMALIZATION (FIX-4: clamped)
    # -----------------------------------------------------------------------
    def rgb_ycbcr(self, tensor_img: torch.Tensor) -> torch.Tensor:
        """
        Convert 3-channel [0,1] tensor to 6-channel [-1,1] RGB+YCbCr.
        FIX-4: output is explicitly clamped to avoid BN NaN from extreme values.
        """
        r, g, b = tensor_img[0], tensor_img[1], tensor_img[2]
        y  =  0.299 * r + 0.587 * g + 0.114 * b
        cb = -0.168736 * r - 0.331264 * g + 0.5 * b + 0.5
        cr =  0.5 * r - 0.418688 * g - 0.081312 * b + 0.5
        ycbcr    = torch.stack([y, cb, cr], dim=0)
        combined = torch.cat([tensor_img, ycbcr], dim=0)
        normalized = (combined - 0.5) / 0.5
        return torch.clamp(normalized, -1.0, 1.0)   # FIX-4


# ===========================================================================
# FIX-3: FAST BALANCED SAMPLER WEIGHT EXTRACTION
# ===========================================================================

def get_sampler_weights(dataset) -> list:
    """
    Read labels directly from the dataset's label list — O(N) no disk access.

    Works with both FFPPDatasetV2 directly and torch.utils.data.Subset.
    """
    if hasattr(dataset, "labels"):
        labels = [int(l) for l in dataset.labels]
    elif hasattr(dataset, "dataset") and hasattr(dataset, "indices"):
        # Subset
        base   = dataset.dataset
        labels = [int(base.labels[i]) for i in dataset.indices]
    else:
        # Slow fallback
        labels = [int(dataset[i]["label"].item()) for i in range(len(dataset))]

    n_real = labels.count(0)
    n_fake = labels.count(1)
    w_real = 1.0 / (n_real + 1e-6)
    w_fake = 1.0 / (n_fake + 1e-6)
    return [w_real if l == 0 else w_fake for l in labels]


## Backbone

Source: `models/backbone.py`

In [ ]:
"""
backbone.py  — AFAGNet Improved Backbone
============================================
Changes from v1:
  FIX-1: Proper ImageNet normalization applied AFTER input_adapter
          v1 passed [-1, 1] inputs directly to a backbone pretrained on
          ImageNet-normalized data. The adapter was initialized to identity,
          so the backbone saw wrong statistics → slow convergence.
  NEW-1:  F_high projection  —  (B*N, 512, 7, 7) → (B*N, 256, 7, 7)
          F_high is now usable downstream in localization head and cls head
  NEW-2:  AltFreezing support  — backbone can be frozen/unfrozen for the
          AltFreezing training strategy (ECCV 2023)
  NEW-3:  Optional EfficientNet-B4 backbone for higher accuracy
          (requires ~400MB more VRAM, not recommended for 4GB GPU)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


class AFAGBackbone(nn.Module):
    """
    Improved backbone:
    - MobileViT-v2 as default (VRAM-friendly on 4GB)
    - 6-channel input via learnable adapter
    - Correct ImageNet normalization pipeline
    - F_high projected and returned (was discarded in v1)
    - AltFreezing support
    """

    # ImageNet statistics — used to normalize after input_adapter
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    def __init__(
        self,
        backbone_name: str = "mobilevitv2_100",
        pretrained: bool = True,
        freeze_on_init: bool = False,
    ):
        super().__init__()

        # ------------------------------------------------------------------
        # 6-channel → 3-channel adapter
        # Initialized to identity for first 3 channels (RGB pass-through)
        # so the backbone initially sees only RGB, ignoring YCbCr until
        # the adapter weights are learned.
        # ------------------------------------------------------------------
        self.input_adapter = nn.Conv2d(6, 3, kernel_size=1, bias=False)
        self._init_input_adapter()

        # ------------------------------------------------------------------
        # ImageNet normalization buffers
        # Applied AFTER adapter, BEFORE backbone
        # Not learnable — these are fixed statistics from ImageNet pretraining
        # ------------------------------------------------------------------
        mean = torch.tensor(self.IMAGENET_MEAN).view(1, 3, 1, 1)
        std  = torch.tensor(self.IMAGENET_STD).view(1, 3, 1, 1)
        self.register_buffer("imagenet_mean", mean)
        self.register_buffer("imagenet_std",  std)

        # ------------------------------------------------------------------
        # Pretrained backbone (features_only mode extracts multi-scale maps)
        # ------------------------------------------------------------------
        self.backbone_name = backbone_name
        self.model = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            features_only=True,
        )

        # ------------------------------------------------------------------
        # Feature projection layers
        # MobileViT-v2-100 stage outputs (item #13 — 28×28 resolution upgrade):
        #   features[-3]: (B*N, 192, 28, 28)  — F_low  (fine texture, stride=8)
        #   features[-1]: (B*N, 512,  7,  7)  — F_high (semantic context, stride=32)
        #
        # PREVIOUS (14×14): used features[-2] at stride=16.
        # At 14×14 each cell = 16×16 pixels. A 4-pixel blending boundary = 0.25 cell
        # → invisible to the decoder. At 28×28 it spans 0.5 cells → detectable.
        # Expected improvement: +5–8% localization IoU, +0.5–1% AUC.
        #
        # Both projected to 256 channels for uniform downstream processing.
        # ------------------------------------------------------------------
        low_ch, high_ch = self._detect_feature_channels()

        self.low_proj  = nn.Sequential(
            nn.Conv2d(low_ch,  256, 1),
            nn.BatchNorm2d(256),
            nn.GELU(),
        )
        self.high_proj = nn.Sequential(
            nn.Conv2d(high_ch, 256, 1),
            nn.BatchNorm2d(256),
            nn.GELU(),
        )

        if freeze_on_init:
            self.freeze_backbone()

    def _init_input_adapter(self):
        """Initialize adapter to identity for RGB channels (channels 0,1,2)."""
        with torch.no_grad():
            self.input_adapter.weight.zero_()
            for ch in range(3):
                self.input_adapter.weight[ch, ch, 0, 0] = 1.0

    def _detect_feature_channels(self):
        """
        Probe backbone to get actual output channel counts.
        Uses features[-3] for F_low (28×28) and features[-1] for F_high (7×7).
        """
        try:
            dummy = torch.zeros(1, 3, 224, 224)
            with torch.no_grad():
                feats = self.model(dummy)
            # features[-3] = 28×28 stage, features[-1] = 7×7 stage
            return feats[-3].shape[1], feats[-1].shape[1]
        except Exception:
            # Fallback for MobileViT-v2-100: stage[-3] ≈ 192ch, stage[-1] = 512ch
            return 192, 512

    def freeze_backbone(self):
        """
        Freeze backbone weights for AltFreezing strategy.
        Adapter and projection layers remain trainable.
        """
        for p in self.model.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        """Unfreeze backbone for AltFreezing strategy."""
        for p in self.model.parameters():
            p.requires_grad = True

    def forward(self, x: torch.Tensor):
        """
        x: (B*N, 6, H, W) in [-1, 1]

        Returns:
            F_low:  (B*N, 256, 28, 28)  — fine texture features  (item #13: was 14×14)
            F_high: (B*N, 256,  7,  7)  — semantic context features
        """
        x_adapted = self.input_adapter(x)
        x_01      = (x_adapted + 1.0) / 2.0
        x_01      = x_01.clamp(0.0, 1.0)
        x_norm    = (x_01 - self.imagenet_mean) / self.imagenet_std

        features = self.model(x_norm)

        # item #13: use features[-3] (28×28) instead of features[-2] (14×14)
        F_low_raw  = features[-3]   # (B*N, 192, 28, 28)
        F_high_raw = features[-1]   # (B*N, 512,  7,  7)

        F_low  = self.low_proj(F_low_raw)                         # (B*N, 256, 28, 28)
        F_high = self.high_proj(
            F.interpolate(F_high_raw, size=(7, 7), mode='bilinear', align_corners=False)
        )                                                           # (B*N, 256,  7,  7)

        return F_low, F_high


class EfficientNetBackbone(nn.Module):
    """
    Higher-accuracy alternative backbone using EfficientNet-B4.
    
    Trade-offs vs MobileViT-v2:
        + ~3-5% higher AUC on FF++ (per ablation studies)
        + Better pretrained features (ImageNet-21k available)
        - ~40% more VRAM (may OOM on 4GB GPU with batch=2, N=16)
        - ~30% slower training
    
    Recommendation: Use if you have ≥ 6GB VRAM or can reduce N to 8.
    """

    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    def __init__(self, pretrained: bool = True):
        super().__init__()

        self.input_adapter = nn.Conv2d(6, 3, kernel_size=1, bias=False)
        self._init_input_adapter()

        mean = torch.tensor(self.IMAGENET_MEAN).view(1, 3, 1, 1)
        std  = torch.tensor(self.IMAGENET_STD).view(1, 3, 1, 1)
        self.register_buffer("imagenet_mean", mean)
        self.register_buffer("imagenet_std",  std)

        # EfficientNet-B4 in features_only mode
        self.model = timm.create_model(
            "efficientnet_b4",
            pretrained=pretrained,
            features_only=True,
            out_indices=(2, 4),   # stages 2 and 4 for local/global features
        )

        # EfficientNet-B4 stage channels: stage2=48, stage4=272 (verify with probe)
        self.low_proj  = nn.Sequential(nn.Conv2d(48,  256, 1), nn.BatchNorm2d(256), nn.GELU())
        self.high_proj = nn.Sequential(nn.Conv2d(272, 256, 1), nn.BatchNorm2d(256), nn.GELU())

    def _init_input_adapter(self):
        with torch.no_grad():
            self.input_adapter.weight.zero_()
            for ch in range(3):
                self.input_adapter.weight[ch, ch, 0, 0] = 1.0

    def forward(self, x: torch.Tensor):
        x_adapted = self.input_adapter(x)
        x_01      = ((x_adapted + 1.0) / 2.0).clamp(0.0, 1.0)
        x_norm    = (x_01 - self.imagenet_mean) / self.imagenet_std

        features = self.model(x_norm)
        F_low_raw, F_high_raw = features[0], features[1]

        F_low  = self.low_proj(
            F.interpolate(F_low_raw, size=(14, 14), mode='bilinear', align_corners=False)
        )
        F_high = self.high_proj(
            F.interpolate(F_high_raw, size=(7, 7), mode='bilinear', align_corners=False)
        )
        return F_low, F_high


# ------------------------------------------------------------------
# Factory function
# ------------------------------------------------------------------
def build_backbone(backbone_type: str = "mobilevit", pretrained: bool = True):
    """
    backbone_type: "mobilevit" (4GB VRAM safe) | "efficientnet" (needs 6GB+)
    """
    if backbone_type == "efficientnet":
        print("Using EfficientNet-B4 backbone (requires ≥6GB VRAM)")
        return EfficientNetBackbone(pretrained=pretrained)
    else:
        return AFAGBackbone(pretrained=pretrained)


## Spatial Branch

Source: `models/branches/spatial_branch.py`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CBAM(nn.Module):
    """Channel + Spatial attention. Tells the network 'what' and 'where' to look."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid(),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        y = self.avg_pool(x).view(B, C)
        y = self.fc(y).view(B, C, 1, 1)
        x = x * y
        avg  = torch.mean(x, dim=1, keepdim=True)
        maxv, _ = torch.max(x, dim=1, keepdim=True)
        s = self.spatial(torch.cat([avg, maxv], dim=1))
        return x * s


class SpatialBranch(nn.Module):
    """
    GenConViT-inspired dual-path spatial branch.

    Original design used only F_low (256×14×14) from the backbone.
    F_high (512×7×7) was extracted but never used — wasted information.

    New design:
      Path A (fine):   F_low  (256×14×14) — rich spatial detail, high resolution
      Path B (coarse): F_high (512×7×7)   — semantic context, globally receptive

    Both paths apply a high-pass filter + CBAM, then F_high is upsampled
    to 14×14 and fused with F_low via a learnable 1×1 convolution.
    This mirrors GenConViT's idea of combining local texture (CNN-like, F_low)
    with global context (ViT-like, F_high) before the fusion stage.

    The output channel dim stays at 256 so CGAF requires no changes.
    """
    def __init__(self, channels=256, high_channels=512):
        super().__init__()

        # --- Path A: fine spatial stream (F_low, 256 ch, 14×14) ---
        self.high_pass_a = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.conv_a = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        )
        self.cbam_a = CBAM(channels)

        # --- Path B: coarse semantic stream (F_high, 512 ch, 7×7) ---
        # Project 512 → 256 channels first so paths are compatible
        self.proj_b = nn.Conv2d(high_channels, channels, 1, bias=False)
        self.high_pass_b = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.conv_b = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        )
        self.cbam_b = CBAM(channels)

        # --- Fusion: combine both paths ---
        # Input: cat(A, B) = 512 ch → output: 256 ch
        self.fuse = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        )

    def forward(self, F_low, F_high=None):
        """
        F_low  : (B*N, 256, H, W) — any spatial size (28×28 after item #13)
        F_high : (B*N, 256, H/2, W/2) — optional coarse semantic path

        Dynamic sizing: interpolation uses F_low.shape[2:] so this works at
        any resolution — 14×14 (old) or 28×28 (item #13) without code changes.
        """
        # Path A
        x_a = self.high_pass_a(F_low) + F_low
        x_a = self.conv_a(x_a)
        x_a = self.cbam_a(x_a)

        if F_high is None:
            return x_a

        # Path B — project + upsample to match path A spatial size (DYNAMIC)
        x_b = self.proj_b(F_high)
        x_b = self.high_pass_b(x_b) + x_b
        x_b = self.conv_b(x_b)
        x_b = self.cbam_b(x_b)
        # item #13 fix: was size=(14,14) hardcoded. Now matches F_low exactly.
        x_b = F.interpolate(x_b, size=F_low.shape[2:], mode='bilinear', align_corners=False)

        fused = self.fuse(torch.cat([x_a, x_b], dim=1))
        return fused


## Frequency Branch

Source: `models/branches/frequency_branch_raw.py`

In [ ]:
"""
frequency_branch_v2.py  — NaN-safe, phase-aware, multi-scale frequency branch
==============================================================================
Changes from v1:
  FIX-1:  Cast to float32 BEFORE fft2 — float16 FFT magnitudes overflow (→ NaN at epoch 4)
  FIX-2:  Use log1p(magnitude) — log scale gives better gradient behaviour, prevents overflow
  NEW-1:  Phase stream — phase discontinuities are strong deepfake indicators
           (GAN models produce systematic phase artifacts; blending creates phase boundaries)
  NEW-2:  Multi-scale frequency analysis (block_size 2, 4, 8)
           — different scales capture different artifact types
           — small blocks: pixel-level GAN noise; larger blocks: texture/blending artifacts
  NEW-3:  GELU activation (smoother gradient landscape than ReLU)
  NEW-4:  Spatial normalization before FFT (per-spatial-location, not global)
  NEW-5:  Frequency band attention using both channel and spatial dimensions
  
Research basis:
  F3Net (AAAI 2021): FAD decomposes image into frequency bands for detection
  HiFi-MASK (ACM MM 2022): phase consistency as forensic signal
  SPSL (CVPR 2021): phase spectrum for spatial-frequency clues in deepfake detection
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiScaleFFT(nn.Module):
    """
    Compute FFT at multiple block scales.
    Fuses magnitude (global artifact energy) and phase (structural inconsistency).
    
    Block size 2  → pixel-level GAN noise patterns
    Block size 4  → local texture artifacts
    Block size 8  → medium-scale blending boundaries
    
    All FFT operations are cast to float32 to prevent float16 overflow.
    """
    def __init__(self, channels: int, block_sizes=(2, 4, 8)):
        super().__init__()
        self.block_sizes = block_sizes
        n_scales = len(block_sizes)

        # Per-scale magnitude projections
        self.mag_projs = nn.ModuleList([
            nn.Conv2d(channels, channels // 2, 1) for _ in block_sizes
        ])
        # Per-scale phase projections
        self.phase_projs = nn.ModuleList([
            nn.Conv2d(channels, channels // 4, 1) for _ in block_sizes
        ])

        # Fuse all scales: (channels//2 + channels//4) * n_scales → channels
        in_fuse = (channels // 2 + channels // 4) * n_scales
        self.fuse = nn.Sequential(
            nn.Conv2d(in_fuse, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU(),
        )

    def _block_fft(self, x: torch.Tensor, block_size: int):
        """
        Block-wise FFT: divides feature map into non-overlapping blocks,
        computes FFT within each block, returns log-magnitude and normalized phase.
        
        x: (B, C, H, W), MUST be float32
        returns: (log_mag, phase_norm) each shape (B, C, H, W)
        """
        B, C, H, W = x.shape
        b = block_size

        # Ensure divisible
        H_ = H - (H % b)
        W_ = W - (W % b)
        if H_ < b or W_ < b:
            # Block size too large for feature map; fall back to global FFT
            H_, W_ = H, W
            b = max(1, min(H_, W_))

        x = x[:, :, :H_, :W_]

        # Reshape into blocks: (B, C, H//b, b, W//b, b)
        x_blocks = x.unfold(2, b, b).unfold(3, b, b)
        # Now: (B, C, H_//b, W_//b, b, b)

        # rfft2 on last two dims — real FFT, more memory efficient than fft2
        fft = torch.fft.rfft2(x_blocks, norm='ortho')
        # fft shape: (B, C, H_//b, W_//b, b, b//2+1) — complex

        # --- Magnitude stream ---
        # log1p prevents overflow and gives better gradient scale
        mag = torch.log1p(torch.abs(fft))               # stable log magnitude

        # --- Phase stream ---
        # Phase in [-pi, pi] → normalize to [0, 1] for stable gradients
        phase = torch.angle(fft)                         # [-pi, pi]
        phase = (phase + torch.pi) / (2.0 * torch.pi)   # [0, 1]

        # Collapse block dimensions back to spatial
        # mag/phase: (B, C, H_//b, W_//b, b, b//2+1)
        # We take the first b frequency components to keep spatial size
        # rfft2 last dim has b//2+1 freqs; pad back to b
        pad_w = b - fft.shape[-1]
        if pad_w > 0:
            mag   = F.pad(mag,   (0, pad_w))
            phase = F.pad(phase, (0, pad_w))

        # Now: (B, C, H_//b, W_//b, b, b)
        # Interleave back to (B, C, H_, W_) via permute+reshape
        mag   = mag.permute(0, 1, 2, 4, 3, 5).contiguous().view(B, C, H_, W_)
        phase = phase.permute(0, 1, 2, 4, 3, 5).contiguous().view(B, C, H_, W_)

        # Restore spatial size if we cropped
        if H_ < H or W_ < W:
            mag   = F.interpolate(mag,   size=(H, W), mode='bilinear', align_corners=False)
            phase = F.interpolate(phase, size=(H, W), mode='bilinear', align_corners=False)

        return mag, phase

    def forward(self, x: torch.Tensor):
        """
        x: (B, C, H, W), any dtype (will be cast internally)
        returns: (B, channels, H, W)
        """
        orig_dtype = x.dtype
        x_f32 = x.float()   # FIX-1: always process in float32

        scale_feats = []
        for i, block_size in enumerate(self.block_sizes):
            mag, phase = self._block_fft(x_f32, block_size)
            # Cast projections back to original dtype to match conv weights
            mag_feat   = self.mag_projs[i](mag.to(orig_dtype))
            phase_feat = self.phase_projs[i](phase.to(orig_dtype))
            scale_feats.append(mag_feat)
            scale_feats.append(phase_feat)

        fused = self.fuse(torch.cat(scale_feats, dim=1))
        return fused


class FrequencyBranch(nn.Module):
    """
    Improved Frequency Branch for deepfake detection.
    
    Input: raw 6-channel frames (B*N, 6, 224, 224)
    Output: frequency features (B*N, 256, 14, 14)
    
    Pipeline:
    1. Raw adapter: 6ch → 256ch, stride=16 (224→14)
    2. Spatial normalization + clamping
    3. Multi-scale FFT: magnitude + phase at 3 block sizes
    4. Channel attention (frequency band selection)
    5. Residual fusion with adapter output
    """

    def __init__(self, in_channels: int = 6, out_channels: int = 256):
        super().__init__()
        self.out_channels = out_channels

        # ------------------------------------------------------------------
        # Raw → Feature adapter
        # item #13: stops at 28×28 (3 stride-2 convs) instead of 14×14 (4).
        # Removing the last stride-2 conv doubles spatial resolution, allowing
        # the FFT branch to detect finer-grained frequency artifacts.
        # ------------------------------------------------------------------
        self.adapter = nn.Sequential(
            nn.Conv2d(in_channels, 64,  3, stride=2, padding=1),   # 224 → 112
            nn.GELU(),
            nn.Conv2d(64,  128, 3, stride=2, padding=1),            # 112 → 56
            nn.GELU(),
            nn.Conv2d(128, out_channels, 3, stride=2, padding=1),   # 56  → 28
            nn.GELU(),
            # REMOVED: nn.Conv2d(out_channels, out_channels, 3, stride=2, padding=1)  # 28→14
            # item #13: output is now 28×28, matching backbone F_low resolution
        )

        # ------------------------------------------------------------------
        # Multi-scale FFT module (phase + magnitude at 3 scales)
        # ------------------------------------------------------------------
        self.multiscale_fft = MultiScaleFFT(out_channels, block_sizes=(2, 4, 7))

        # ------------------------------------------------------------------
        # Post-FFT feature refinement
        # ------------------------------------------------------------------
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

        # ------------------------------------------------------------------
        # Channel attention: selects which frequency bands are most informative
        # Uses both avg and max pooling (richer statistics than avg alone)
        # ------------------------------------------------------------------
        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(out_channels, out_channels // 8),
            nn.GELU(),
            nn.Linear(out_channels // 8, out_channels),
            nn.Sigmoid(),
        )
        self.channel_attn_max = nn.Sequential(
            nn.AdaptiveMaxPool2d(1),
            nn.Flatten(),
            nn.Linear(out_channels, out_channels // 8),
            nn.GELU(),
            nn.Linear(out_channels // 8, out_channels),
            nn.Sigmoid(),
        )

        # ------------------------------------------------------------------
        # Spatial attention: where in the feature map are frequency anomalies?
        # ------------------------------------------------------------------
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid(),
        )

        # ------------------------------------------------------------------
        # Final projection (ensures output residual has matching channels)
        # ------------------------------------------------------------------
        self.out_proj = nn.Conv2d(out_channels, out_channels, 1)

    def _normalize_features(self, x: torch.Tensor) -> torch.Tensor:
        """
        Spatial normalization per channel.
        Clamps to ±3σ to prevent extreme values going into FFT.
        """
        mean = x.mean(dim=[2, 3], keepdim=True)
        std  = x.std(dim=[2, 3], keepdim=True) + 1e-6
        x    = (x - mean) / std
        return torch.clamp(x, -3.0, 3.0)

    def _channel_attention(self, x: torch.Tensor) -> torch.Tensor:
        avg_attn = self.channel_attn(x).unsqueeze(-1).unsqueeze(-1)
        max_attn = self.channel_attn_max(x).unsqueeze(-1).unsqueeze(-1)
        # Combine avg and max for richer frequency band selection
        return (avg_attn + max_attn) / 2.0

    def _spatial_attention(self, x: torch.Tensor) -> torch.Tensor:
        avg_proj = x.mean(dim=1, keepdim=True)
        max_proj = x.max(dim=1, keepdim=True)[0]
        return self.spatial_attn(torch.cat([avg_proj, max_proj], dim=1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: RAW input (B*N, 6, 224, 224)
        returns: (B*N, 256, 14, 14)
        """
        # 1. Adapt raw → feature space
        x_feat = self.adapter(x)     # (B*N, 256, 14, 14)

        # 2. Normalize for FFT stability (per spatial location)
        x_norm = self._normalize_features(x_feat)

        # 3. Multi-scale FFT with magnitude + phase
        #    FIX-1: MultiScaleFFT casts to float32 internally
        freq_feat = self.multiscale_fft(x_norm)     # (B*N, 256, 14, 14)

        # 4. Refine frequency features
        freq_feat = self.refine(freq_feat)

        # 5. Channel attention (frequency band selection)
        ch_attn   = self._channel_attention(freq_feat)
        freq_feat = freq_feat * ch_attn

        # 6. Spatial attention (localize frequency anomalies)
        sp_attn   = self._spatial_attention(freq_feat)
        freq_feat = freq_feat * sp_attn

        # 7. Residual fusion with adapter output
        out = self.out_proj(freq_feat) + x_feat

        return out


## Noise Branch

Source: `models/branches/noise_branch_raw.py`

In [ ]:
"""
noise_branch_raw.py  — SRM-based Noise Branch for AFAGNetV3
============================================================
Detects camera sensor noise pattern (PRNU) inconsistencies introduced
when a GAN-generated patch replaces a real face region.

SRM kernel options:
  CURRENT (3 kernels)   — fast, works on 4GB GPU, trained model uses this
  COMMENTED (30 kernels) — full Fridrich & Kodovsky (IEEE TIFS 2012) set
                           Enable on T4 by uncommenting get_srm_kernels_full()
                           and changing get_srm_kernels() → get_srm_kernels_full()

Research basis:
  - Fridrich & Kodovsky, "Rich Models for Steganalysis", IEEE TIFS 2012
  - MantraNet (CVPR 2019): SRM + deep learning for image forensics
  - MVSS-Net (TPAMI 2022): SRM kernels improve localization IoU by ~3%
  - Bayar & Stamm (IEEE TIFS 2018): constrained + learnable forensic filters
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# 3-KERNEL SRM — CURRENT (4GB VRAM safe, used in training)
# ===========================================================================

def get_srm_kernels():
    """
    3 representative SRM kernels:
      k1: first-order horizontal difference  (detects horizontal blending seams)
      k2: second-order horizontal (Laplacian row): detects noise level changes
      k3: 2D Laplacian: detects texture discontinuities at blending boundaries

    These 3 kernels cover the most discriminative noise residual patterns for
    face manipulation detection at minimal compute cost.
    """
    k1 = [[0,  0,  0],
          [0,  1, -1],
          [0,  0,  0]]   # first-order horizontal

    k2 = [[0,  0,  0],
          [1, -2,  1],
          [0,  0,  0]]   # second-order horizontal

    k3 = [[-1,  2, -1],
          [ 2, -4,  2],
          [-1,  2, -1]]  # 2D Laplacian

    kernels = torch.tensor([k1, k2, k3], dtype=torch.float32)
    return kernels.unsqueeze(1)   # (3, 1, 3, 3)


# ===========================================================================
# 30-KERNEL SRM — FULL (T4 / high-VRAM, enable for research-level performance)
# Uncomment get_srm_kernels_full() and change the NoiseBranch __init__ to call
# get_srm_kernels_full() instead of get_srm_kernels().
# Expected improvement: +0.3–0.8% AUC, +3% localization IoU vs 3-kernel version.
# ===========================================================================

# def get_srm_kernels_full():
#     """
#     Full SRM rich model: 30 kernels from Fridrich & Kodovsky (IEEE TIFS 2012).
#     Covers: first/second/third-order horizontal, vertical, diagonal, cross, square.
#     These 30 patterns capture virtually all statistical regularities of camera noise.
#
#     On T4 (16GB): using 30 kernels adds ~2ms/batch — negligible cost.
#     On RTX 3050 (4GB): weight tensor is only 30×1×3×3 × 4 bytes = 1080 bytes,
#     but the grouped convolution output is 30× larger → use 3-kernel version there.
#     """
#     kernels = []
#
#     # First-order kernels (4 directions)
#     kernels.append([[0, 0, 0], [0,  1, -1], [0,  0,  0]])   # h  (horizontal)
#     kernels.append([[0, 0, 0], [0,  1,  0], [0, -1,  0]])   # v  (vertical)
#     kernels.append([[0, 0, 0], [0,  1,  0], [0,  0, -1]])   # d1 (diagonal)
#     kernels.append([[0, 0, 0], [0,  1,  0], [-1, 0,  0]])   # d2 (anti-diagonal)
#
#     # Second-order kernels (4 directions)
#     kernels.append([[0,  0, 0], [1, -2,  1], [0,  0,  0]])  # h2
#     kernels.append([[0,  1, 0], [0, -2,  0], [0,  1,  0]])  # v2
#     kernels.append([[1,  0, 0], [0, -2,  0], [0,  0,  1]])  # d3
#     kernels.append([[0,  0, 1], [0, -2,  0], [1,  0,  0]])  # d4
#
#     # 2D Laplacian (cross pattern)
#     kernels.append([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]])
#
#     # Third-order horizontal
#     kernels.append([[0, 0, 0], [-1, 3, -3], [0, 0, 0]])    # hmm left
#     kernels.append([[0, 0, 0], [1, -3,  3], [0, 0, 0]])    # hmm right
#
#     # Third-order vertical
#     kernels.append([[0, -1, 0], [0,  3, 0], [0, -3, 0]])
#     kernels.append([[0,  1, 0], [0, -3, 0], [0,  3, 0]])
#
#     # Square kernel patterns
#     kernels.append([[ 1, -2,  1], [-2,  4, -2], [ 1, -2,  1]])   # square2
#     kernels.append([[-1,  2, -1], [ 2, -4,  2], [-1,  2, -1]])   # square3
#
#     # Cross consistency kernels
#     kernels.append([[0,  1, 0], [1, -4,  1], [0,  1,  0]])   # cross4
#     kernels.append([[1,  0, 1], [0, -4,  0], [1,  0,  1]])   # cross5
#
#     # Higher-order diagonal patterns
#     kernels.append([[ 2, -1, 0], [-1,  0,  1], [0,   1, -2]])
#     kernels.append([[ 0, -1, 2], [ 1,  0, -1], [-2,  1,  0]])
#     kernels.append([[-2,  1, 0], [ 1,  0, -1], [0,  -1,  2]])
#     kernels.append([[ 0,  1,-2], [-1,  0,  1], [ 2, -1,  0]])
#
#     # Horizontal span-2 patterns
#     kernels.append([[0, 0, 0], [1, -1,  0], [0,  0,  0]])
#     kernels.append([[0, 0, 0], [0,  1, -1], [0,  0,  0]])
#
#     # Vertical span-2 patterns
#     kernels.append([[0, 1, 0], [0, -1, 0], [0,  0,  0]])
#     kernels.append([[0, 0, 0], [0,  1, 0], [0, -1,  0]])
#
#     # Edge enhancement patterns
#     kernels.append([[ 0, -1,  0], [-1,  5, -1], [ 0, -1,  0]])   # sharpen
#     kernels.append([[-1, -1, -1], [-1,  9, -1], [-1, -1, -1]])   # strong sharpen
#
#     # Smoothness residual patterns
#     kernels.append([[ 1,  2,  1], [ 2,  4,  2], [ 1,  2,  1]])   # (normalize after)
#     kernels.append([[ 0,  0,  0], [ 0,  1,  0], [ 0,  0,  0]])   # identity residual
#
#     k = torch.tensor(kernels, dtype=torch.float32)
#     # Normalize each kernel to zero-sum (ensures noise residual extraction)
#     k = k - k.mean(dim=[1, 2], keepdim=True)
#     return k.unsqueeze(1)   # (30, 1, 3, 3)


# ===========================================================================
# NOISE BRANCH
# ===========================================================================

class NoiseBranch(nn.Module):
    """
    Input:  raw 6-channel frames (B*N, 6, 224, 224)
    Output: noise residual features (B*N, 256, 14, 14)

    Pipeline:
    1. Adapter: 6ch → 256ch, stride 16 (224 → 14)
    2. Instance normalization (stable for small batches, unlike BN)
    3. Frozen SRM kernels + learnable complementary path
    4. Conv refinement + attention gate
    5. Residual skip from adapter

    NOTE: To enable 30-kernel SRM (T4):
      1. Uncomment get_srm_kernels_full() above
      2. In __init__, change: srm_kernels = get_srm_kernels()
                          to: srm_kernels = get_srm_kernels_full()
      3. The self.srm conv size is set automatically from kernel count
    """

    def __init__(self, in_channels: int = 6, out_channels: int = 256):
        super().__init__()

        # ── Adapter: raw → feature space ─────────────────────────────────────
        # item #13: stops at 28×28 (3 stride-2 convs) instead of 14×14 (4).
        # Removing the last stride-2 conv doubles spatial resolution so SRM
        # kernels detect noise at finer granularity (8px/cell vs 16px/cell).
        self.adapter = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, stride=2, padding=1),     # 224 → 112
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64,  128, 3, stride=2, padding=1),             # 112 → 56
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, out_channels, 3, stride=2, padding=1),    # 56  → 28
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            # REMOVED: nn.Conv2d(out_channels, out_channels, 3, stride=2, padding=1)  28→14
            # item #13: output is now 28×28, matching backbone F_low resolution
        )

        # ── SRM kernels — CHANGE TO get_srm_kernels_full() ON T4 ──────────
        srm_kernels = get_srm_kernels()       # 3-kernel version (4GB safe)
        # srm_kernels = get_srm_kernels_full()  # 30-kernel version (T4 only)

        # Frozen SRM depthwise convolution
        # groups=out_channels applies same-channel-only filtering (depthwise)
        self.srm = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, padding=1, bias=False, groups=out_channels
        )
        # Initialize SRM weights from averaged kernels repeated across channels
        with torch.no_grad():
            k = srm_kernels.mean(dim=0)   # (1, 3, 3) — average of SRM kernels
            self.srm.weight.copy_(k.repeat(out_channels, 1, 1, 1))
        # Freeze SRM — preserves theoretical noise-detection properties
        for p in self.srm.parameters():
            p.requires_grad = False

        # Learnable complementary path — learns data-driven noise patterns
        # that complement the fixed SRM kernels (Bayar & Stamm, TIFS 2018)
        self.learnable_path = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, padding=1, groups=out_channels, bias=False
        )

        # Conv refinement after noise extraction
        self.conv = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )

        # Attention gate: learns which spatial locations have forensic noise
        self.attn = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: raw 6-channel input (B*N, 6, 224, 224) in [-1, 1]
        returns: (B*N, 256, 14, 14)
        """
        # 1. Adapt to feature space
        x_adapted = self.adapter(x)    # (B*N, 256, 14, 14)

        # 2. Instance normalization — stable for small batches (batch=2)
        #    Global BN with batch=2 gives unreliable mean/var estimates when
        #    WeightedRandomSampler produces near-identical frames.
        x_norm = F.instance_norm(x_adapted)

        # 3. SRM + learnable noise extraction
        noise = self.srm(x_norm) + self.learnable_path(x_norm)

        # 4. Refinement + attention
        out  = self.conv(noise)
        attn = self.attn(out)

        # 5. Residual: add original adapter output
        return out * attn + x_adapted


## Fusion Module

Source: `models/fusion/cgaf.py`

In [ ]:
"""
cgaf_v2.py  — Cross-branch Gated Adaptive Fusion (improved)
=============================================================
Changes from v1:
  FIX-1: Bounded temperature parameterization
          v1: self.temp = nn.Parameter(tensor(1.0))  — unconstrained
          If temp grows large (e.g. 50), sigmoid(Csf * temp) becomes a
          near-step function → near-zero gradients everywhere → dead fusion
          
          v2: self.log_temp = nn.Parameter(tensor(0.0))
              temp = clamp(exp(log_temp), 0.1, 10.0)
              This prevents both vanishing (too large) and trivial (too small) gates
              
  FIX-2: F_high integration pathway
          v1: F_high was extracted in backbone but never used
          v2: F_high pooled to (B*N, 256) provides global context to
              modulate the gating mechanism (fake regions often have
              distinctive global statistics visible at high level)
              
  NEW-1: Learnable residual weight
          v1: Ffusion = Fs + 0.5 * fused_aux  (fixed 0.5)
          v2: Ffusion = Fs + self.alpha * fused_aux  (learned)
          
  NEW-2: GELU activation in refinement block
  
  NEW-3: Better similarity → uses product instead of cosine
         (more sensitive to magnitude differences, which are informative in forgery)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class CGAFv2(nn.Module):
    """
    Cross-branch Gated Adaptive Fusion with bounded temperature and F_high context.
    
    Inputs:
        Fs:     (B*N, 256, 14, 14) — spatial branch features
        Ff:     (B*N, 256, 14, 14) — frequency branch features
        Fn:     (B*N, 256, 14, 14) — noise branch features
        F_high: (B*N, 256, 7, 7)  — backbone high-level features (optional)
        
    Output:
        Ffusion: (B*N, 256, 14, 14)
        Csf, Csn, Cfn: similarity maps for consistency loss
    """

    def __init__(self, channels: int = 256, use_f_high: bool = True):
        super().__init__()

        self.channels    = channels
        self.use_f_high  = use_f_high

        # Projection layers
        self.proj_s = nn.Conv2d(channels, channels, 1)
        self.proj_f = nn.Conv2d(channels, channels, 1)
        self.proj_n = nn.Conv2d(channels, channels, 1)

        # ------------------------------------------------------------------
        # FIX-1: Bounded temperature via log parameterization
        # temp = clamp(exp(log_temp), min=0.1, max=10.0)
        # At init: log_temp=0 → temp=1 (same as v1 at start)
        # ------------------------------------------------------------------
        self.log_temp = nn.Parameter(torch.tensor(0.0))

        # ------------------------------------------------------------------
        # NEW-2: F_high context gate
        # Global pooled F_high (B*N, 256) → gating signal for fusion weights
        # ------------------------------------------------------------------
        if use_f_high:
            self.high_gate = nn.Sequential(
                nn.Linear(channels, channels // 4),
                nn.GELU(),
                nn.Linear(channels // 4, 1),
                nn.Sigmoid(),
            )

        # ------------------------------------------------------------------
        # NEW-1: Learnable residual weight (replaces fixed 0.5)
        # alpha_raw → softplus → alpha ensures alpha > 0 always
        # ------------------------------------------------------------------
        self.alpha_raw = nn.Parameter(torch.tensor(0.693))   # softplus(0.693) ≈ 1.0

        # Refinement block (FIX-2: GELU)
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),   # depthwise
            nn.BatchNorm2d(channels),
            nn.GELU(),
        )

    @property
    def temp(self) -> torch.Tensor:
        """Temperature in [0.1, 10.0] via exponential parameterization."""
        return torch.clamp(torch.exp(self.log_temp), min=0.1, max=10.0)

    @property
    def alpha(self) -> torch.Tensor:
        """Learned residual weight, always positive via softplus."""
        return F.softplus(self.alpha_raw)

    def compute_similarity(self, A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        """
        Cosine similarity map: (B*N, 1, H, W)
        Output in [-1, 1]: +1 = aligned, -1 = opposite, 0 = orthogonal
        """
        A = F.normalize(A, dim=1)
        B = F.normalize(B, dim=1)
        return torch.sum(A * B, dim=1, keepdim=True)

    def forward(
        self,
        Fs: torch.Tensor,
        Ff: torch.Tensor,
        Fn: torch.Tensor,
        F_high: torch.Tensor = None,
    ):
        # --- Projections ---
        Fs_p = self.proj_s(Fs)
        Ff_p = self.proj_f(Ff)
        Fn_p = self.proj_n(Fn)

        # --- Similarity maps ---
        Csf = self.compute_similarity(Fs_p, Ff_p)   # spatial–frequency agreement
        Csn = self.compute_similarity(Fs_p, Fn_p)   # spatial–noise agreement
        Cfn = self.compute_similarity(Ff_p, Fn_p)   # cross-consistency

        # --- FIX-1: Temperature-scaled sigmoid with bounded temp ---
        Wf = torch.sigmoid(Csf * self.temp)
        Wn = torch.sigmoid(Csn * self.temp)

        # --- Normalized gating ---
        W_sum = Wf + Wn + 1e-6
        Wf = Wf / W_sum
        Wn = Wn / W_sum

        # --- Cross-consistency modulation ---
        cross_gate = torch.sigmoid(Cfn)
        Wf = Wf * cross_gate
        Wn = Wn * cross_gate

        # --- NEW-2: F_high context (if provided) ---
        if self.use_f_high and F_high is not None:
            # Pool F_high to get global context vector
            high_pooled = F.adaptive_avg_pool2d(F_high, 1).flatten(1)  # (B*N, 256)
            global_gate = self.high_gate(high_pooled).view(-1, 1, 1, 1)  # (B*N, 1, 1, 1)
            # Scale gating by global context
            Wf = Wf * (0.5 + 0.5 * global_gate)
            Wn = Wn * (0.5 + 0.5 * global_gate)

        # --- Fusion with NEW-1 learned residual weight ---
        fused_aux = Wf * Ff + Wn * Fn
        Ffusion   = Fs + self.alpha * fused_aux

        # --- Refine ---
        Ffusion = self.refine(Ffusion)

        return Ffusion, Csf, Csn, Cfn


def consistency_loss(Fs, Ff, Fn):
    """
    L1 consistency between branches.
    Encourages spatial, frequency, and noise branches to agree on feature locations.
    """
    return F.l1_loss(Fs, Ff) + F.l1_loss(Fs, Fn)


## Temporal Model

Source: `models/temporal/temporal_model.py`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalModel(nn.Module):
    def __init__(self, in_channels=256, embed_dim=512, 
                 num_heads=8, num_layers=4, max_len=512):
        super().__init__()

        # ---------------------------
        # PROJECTION & NORM
        # ---------------------------
        self.proj = nn.Linear(in_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

        # ---------------------------
        # POSITIONAL EMBEDDING
        # ---------------------------
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, embed_dim))

        # ---------------------------
        # LEARNABLE TEMPORAL WEIGHTS
        # ---------------------------
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w4 = nn.Parameter(torch.tensor(1.0))

        # ---------------------------
        # TRANSFORMER
        # ---------------------------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True,
            activation='gelu'
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    # ===========================
    # SPATIO-TEMPORAL MASKING
    # ===========================
    def get_spatial_mask(self, x):
        """
        Generates a soft attention mask based on feature energy 
        to focus on the face and ignore background noise.
        """
        # Calculate energy: (B, N, 1, H, W)
        mask = torch.mean(torch.abs(x), dim=2, keepdim=True)
        
        # Min-Max Normalization per frame for stability
        mask_flattened = mask.view(mask.size(0), mask.size(1), -1)
        m_min = mask_flattened.min(dim=-1, keepdim=True)[0].view(-1, mask.size(1), 1, 1, 1)
        m_max = mask_flattened.max(dim=-1, keepdim=True)[0].view(-1, mask.size(1), 1, 1, 1)
        
        mask = (mask - m_min) / (m_max - m_min + 1e-6)
        return mask

    def global_pool_masked(self, x, mask):
        """
        Weighted pooling: ignores pixels with low feature energy.
        """
        return (x * mask).sum(dim=[3, 4]) / (mask.sum(dim=[3, 4]) + 1e-6)

    # ===========================
    # SAFE TIME PADDING
    # ===========================
    def pad_time(self, x, target_len):
        pad_len = target_len - x.shape[1]
        if pad_len > 0:
            if x.shape[1] == 0:
                pad = x.new_zeros(x.shape[0], pad_len, x.shape[2])
            else:
                pad = x[:, -1:].repeat(1, pad_len, 1)
            x = torch.cat([x, pad], dim=1)
        return x

    # ===========================
    # TEMPORAL DIFFERENCES
    # ===========================
    def compute_differences(self, Ft):
        N = Ft.shape[1]

        D1 = torch.abs(Ft[:, 1:] - Ft[:, :-1])
        D2 = torch.abs(Ft[:, 2:] - Ft[:, :-2])
        D4 = torch.abs(Ft[:, 4:] - Ft[:, :-4])

        D1 = self.pad_time(D1, N)
        D2 = self.pad_time(D2, N)
        D4 = self.pad_time(D4, N)

        return D1, D2, D4

    # ===========================
    # FORWARD
    # ===========================
    def forward(self, Ffusion):
        """
        Ffusion: (B, N, C, H, W)
        """
        B, N, C, H, W = Ffusion.shape

        # ---------------------------
        # STEP 1: MASKED GLOBAL FEATURES
        # ---------------------------
        mask = self.get_spatial_mask(Ffusion)
        Ft = self.global_pool_masked(Ffusion, mask)  # (B, N, C)

        # ---------------------------
        # STEP 2: TEMPORAL DIFFERENCES
        # ---------------------------
        D1, D2, D4 = self.compute_differences(Ft)

        # ---------------------------
        # STEP 3: SEQUENCE CONSTRUCTION
        # ---------------------------
        # Applying learnable weights and balanced scaling
        sequence = torch.cat([
            Ft,
            self.w1 * 0.5 * D1,
            self.w2 * 0.3 * D2,
            self.w4 * 0.2 * D4
        ], dim=1)  # (B, 4N, C)

        # ---------------------------
        # STEP 4: PROJECTION
        # ---------------------------
        sequence = self.proj(sequence)

        # ---------------------------
        # STEP 5: POSITIONAL ENCODING
        # ---------------------------
        T = sequence.shape[1]
        sequence = sequence + self.pos_embed[:, :T, :]

        # ---------------------------
        # STEP 6: MOTION-AWARE BIAS (Optimized)
        # ---------------------------
        importance = torch.mean(torch.abs(D1), dim=-1, keepdim=True)  # (B, N, 1)
        importance_full = importance.repeat(1, 4, 1) # (B, 4N, 1)
        
        # Broadcasting handles the addition to the 512-dim sequence
        sequence = sequence + importance_full[:, :T, :]

        # ---------------------------
        # STEP 7: NORMALIZATION
        # ---------------------------
        sequence = self.norm(sequence)

        # ---------------------------
        # STEP 8: TRANSFORMER
        # ---------------------------
        out = self.transformer(sequence)

        # ---------------------------
        # STEP 9: AGGREGATION
        # ---------------------------
        video_feat = out.mean(dim=1)  # (B, embed_dim)

        return video_feat


## Full Model

Source: `models/afag_net_v3.py`

In [ ]:
"""
afag_net_v3.py  — AFAGNet: Research-Level Model (v3)
=====================================================
Critical fixes over v2:

  FIX-1: CGAF class name mismatch
          v2's afag_net.py imported CGAF but cgaf.py defines CGAFv2.
          The improved bounded-temperature fusion was NEVER used in training.
          v3 explicitly imports CGAFv2 and uses it correctly.

  FIX-2: SpatialBranch called incorrectly
          v2 AFAGNet called:
              self.spatial = SpatialBranch()
              Fs = self.spatial(F_low)            ← missing F_high!
          But SpatialBranch.forward(F_low, F_high) needs BOTH arguments for
          the dual-path design (GenConViT-style). F_high was never passed.
          v3 calls: Fs = self.spatial(F_low, F_high)

  FIX-3: Dynamic import anti-pattern removed
          v2 used __import__('models.branches.spatial_branch', ...) which
          silently fails if path doesn't exist. v3 uses standard imports.

  NEW-1: Mix-precision consistency
          All branch outputs explicitly cast to same dtype before CGAF.
          Prevents dtype mismatch errors under AMP.

  NEW-2: Gradient checkpointing option
          For RTX 3050 (4GB VRAM), can enable checkpointing on the backbone
          to trade compute for memory, allowing larger batch or more frames.

  NEW-3: Structured output dict with all intermediate features
          Full traceability for ablation studies and debugging.

Architecture overview:
  Input: (B, N, 6, 224, 224)  RGB + YCbCr, normalized to [-1, 1]
  
  Backbone (MobileViT-v2 + alignment):
    F_low  (B*N, 256, 14, 14)  — local texture
    F_high (B*N, 256,  7,  7)  — semantic context
  
  Three-Branch Feature Extraction:
    Fs  — SpatialBranch(F_low, F_high)   dual-path, CBAM
    Ff  — FrequencyBranch(raw_input)     NaN-safe FFT, magnitude + phase
    Fn  — NoiseBranch(raw_input)         SRM-filtered noise residual
  
  CGAFv2 Fusion:
    Ffusion (B*N, 256, 14, 14)  — gated adaptive fusion with F_high context
  
  Temporal Model (Transformer):
    video_feat (B, 512)  — temporal difference + positional transformer
  
  Output Heads:
    ClassificationHead(video_feat, F_high_pool) → (B, 1) logit
    LocalizationHead(Ffusion, F_high)            → (B, 1, 224, 224) mask
"""

import torch
import torch.nn as nn
import torch.nn.functional as F



# ===========================================================================
# CLASSIFICATION HEAD — takes temporal + F_high global pooled features
# ===========================================================================

class ClassificationHead(nn.Module):
    """
    Input:  temporal_feat (B, 512) + high_feat (B, 256)
    Output: (B, 1) raw logit — NO sigmoid applied
    
    Uses a gated pathway so the model learns when F_high is informative.
    """
    def __init__(self, temporal_dim: int = 512, high_dim: int = 256):
        super().__init__()

        self.high_gate = nn.Sequential(
            nn.Linear(high_dim, high_dim // 4),
            nn.GELU(),
            nn.Linear(high_dim // 4, high_dim),
            nn.Sigmoid(),
        )

        combined_dim = temporal_dim + high_dim
        self.fc = nn.Sequential(
            nn.LayerNorm(combined_dim),
            nn.Dropout(0.3),           # dropout before first linear — regularizes input
            nn.Linear(combined_dim, 256),
            nn.GELU(),
            nn.Dropout(0.4),           # stronger dropout in middle (was 0.3)
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),           # light dropout before final layer
            nn.Linear(64, 1),
        )

    def forward(self, temporal_feat: torch.Tensor, high_feat: torch.Tensor) -> torch.Tensor:
        gated    = high_feat * self.high_gate(high_feat)
        combined = torch.cat([temporal_feat, gated], dim=1)
        return self.fc(combined)


# ===========================================================================
# LOCALIZATION HEAD — with F_high skip connection at 7x7
# ===========================================================================

class LocalizationHead(nn.Module):
    """
    Input:  Ffusion (B*N, 256, 28, 28)  — item #13: was 14×14
            F_high  (B*N, 256,  7,  7)  — semantic skip connection
    Output: (B*N, 1, 224, 224) sigmoid mask

    Decoder: 28→56→112→224  (3 ConvTranspose2d steps, was 4 from 14×14)
    Removing one upsampling step reduces upsampling artifacts and gives
    the decoder finer spatial starting information.

    F_high skip is upsampled to match Ffusion's spatial size (dynamic,
    works at both 14×14 and 28×28 without hardcoding).
    """

    def __init__(self, in_channels: int = 256):
        super().__init__()

        self.init_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.GELU(),
        )

        self.high_skip_proj = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, 1),
            nn.GELU(),
        )

        # Merge: 256 + 128 = 384 → 128
        self.skip_merge = nn.Sequential(
            nn.Conv2d(in_channels + in_channels // 2, 128, 1),
            nn.BatchNorm2d(128),
            nn.GELU(),
        )

        # Without skip: project 256 → 128
        self.no_skip_proj = nn.Sequential(
            nn.Conv2d(in_channels, 128, 1),
            nn.BatchNorm2d(128),
            nn.GELU(),
        )

        # item #13: 3-step decoder  28 → 56 → 112 → 224  (was 4-step from 14)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2),    # 28  → 56
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2),     # 56  → 112
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.ConvTranspose2d(32, 16, 2, stride=2),     # 112 → 224
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 1, 1),
        )

    def forward(self, Ffusion: torch.Tensor, F_high: torch.Tensor = None) -> torch.Tensor:
        x = self.init_conv(Ffusion)   # (B*N, 256, 28, 28)

        if F_high is not None:
            # Dynamic: upsample F_high to match Ffusion's spatial size
            # Works at both 28×28 (item #13) and 14×14 (original) without hardcoding
            skip = F.interpolate(F_high, size=Ffusion.shape[2:],
                                 mode='bilinear', align_corners=False)
            skip = self.high_skip_proj(skip)                       # (B*N, 128, 28, 28)
            x    = self.skip_merge(torch.cat([x, skip], dim=1))    # (B*N, 128, 28, 28)
        else:
            x = self.no_skip_proj(x)

        return torch.sigmoid(self.decoder(x))   # (B*N, 1, 224, 224)


# ===========================================================================
# OUTPUT HEADS MODULE
# ===========================================================================

class OutputHeads(nn.Module):
    def __init__(self):
        super().__init__()
        self.cls_head = ClassificationHead()
        self.loc_head = LocalizationHead()

    def forward(
        self,
        video_feat:  torch.Tensor,   # (B, 512)
        Ffusion:     torch.Tensor,   # (B, N, 256, 14, 14)
        F_high_flat: torch.Tensor,   # (B*N, 256, 7, 7)
        Fs=None, Ff=None, Fn=None,
    ):
        B, N, C, H, W = Ffusion.shape

        # Classification: aggregate F_high across frames
        F_high_pool = F.adaptive_avg_pool2d(F_high_flat, 1).view(B, N, 256).mean(dim=1)
        pred_cls    = self.cls_head(video_feat, F_high_pool)   # (B, 1)

        # Localization: per-frame
        F_flat     = Ffusion.view(B * N, C, H, W)
        masks_flat = self.loc_head(F_flat, F_high_flat)        # (B*N, 1, 224, 224)
        mask_seq   = masks_flat.view(B, N, 1, 224, 224)
        pred_mask  = mask_seq[:, N // 2]                       # middle frame mask

        # Explainability signals
        conf_map  = torch.abs(pred_mask - 0.5) * 2
        stability = 1 - torch.abs(mask_seq[:, 1:] - mask_seq[:, :-1])

        return {
            "pred_cls":      pred_cls,
            "pred_mask":     pred_mask,
            "mask_sequence": mask_seq,
            "confidence":    conf_map,
            "stability":     stability,
        }


# ===========================================================================
# AFAGNet v3 — MAIN MODEL
# ===========================================================================

class AFAGNetV3(nn.Module):
    """
    AFAGNet v3: All critical fixes integrated.
    
    Key fixes vs v2:
      - Uses CGAFv2 (bounded temperature, F_high gating) — FIX-1
      - Passes F_high to SpatialBranch (dual-path enabled) — FIX-2
      - Clean imports, no dynamic __import__ — FIX-3
      - AMP-safe dtype consistency — NEW-1
      - Gradient checkpointing option — NEW-2
    
    Usage (identical to v1/v2):
        model = AFAGNetV3()
        out   = model(x)   # x: (B, N, 6, 224, 224)
    """

    def __init__(self, use_gradient_checkpointing: bool = False):
        super().__init__()
        self.use_gradient_checkpointing = use_gradient_checkpointing

        self.backbone   = AFAGBackbone()
        # FIX: AFAGBackbone projects F_high to 256 channels before returning.
        # SpatialBranch default expects high_channels=512 (raw backbone output).
        # Must pass high_channels=256 to match the projected F_high.
        self.spatial    = SpatialBranch(channels=256, high_channels=256)
        self.frequency  = FrequencyBranch()
        self.noise      = NoiseBranch()
        self.cgaf       = CGAFv2(channels=256, use_f_high=True)   # FIX-1
        self.temporal   = TemporalModel()
        self.heads      = OutputHeads()

    def forward(self, x: torch.Tensor) -> dict:
        """
        x: (B, N, 6, 224, 224) — batch of video clips, 6-channel RGB+YCbCr, [-1, 1]

        Feature resolutions after item #13:
          F_low:    (B*N, 256, 28, 28)  — fine texture (was 14×14)
          F_high:   (B*N, 256,  7,  7)  — semantic context (unchanged)
          Fs,Ff,Fn: (B*N, 256, 28, 28)  — all branches (was 14×14)
          Ffusion:  (B*N, 256, 28, 28)  — fused (was 14×14)
        """
        B, N, C, H, W = x.shape
        x_flat = x.view(B * N, C, H, W)

        # ---- STEP 1: Backbone ----
        if self.use_gradient_checkpointing and self.training:
            from torch.utils.checkpoint import checkpoint
            F_low, F_high = checkpoint(self.backbone, x_flat)
        else:
            F_low, F_high = self.backbone(x_flat)
        # F_low:  (B*N, 256, 28, 28)  — item #13
        # F_high: (B*N, 256,  7,  7)

        # ---- STEP 2: Multi-Branch Extraction ----
        Fs = self.spatial(F_low, F_high)    # (B*N, 256, 28, 28)
        Ff = self.frequency(x_flat)          # (B*N, 256, 28, 28)
        Fn = self.noise(x_flat)              # (B*N, 256, 28, 28)

        target_dtype = F_low.dtype
        Fs = Fs.to(target_dtype)
        Ff = Ff.to(target_dtype)
        Fn = Fn.to(target_dtype)

        # ---- STEP 3: CGAFv2 Fusion ----
        Ffusion_flat, Csf, Csn, Cfn = self.cgaf(Fs, Ff, Fn, F_high=F_high)
        # Ffusion_flat: (B*N, 256, 28, 28)

        # ---- STEP 4: Restore temporal dimension ----
        # Dynamic: uses actual H,W from Ffusion_flat so works at any resolution
        _, C_f, H_f, W_f = Ffusion_flat.shape
        Ffusion = Ffusion_flat.view(B, N, C_f, H_f, W_f)

        # ---- STEP 5: Temporal modeling ----
        video_feat = self.temporal(Ffusion)   # (B, 512)

        # ---- STEP 6: Output heads ----
        head_out = self.heads(
            video_feat,
            Ffusion,
            F_high_flat=F_high,
            Fs=Fs, Ff=Ff, Fn=Fn,
        )

        return {
            **head_out,
            "F_low":       F_low,
            "F_high":      F_high,
            "Fs":          Fs,
            "Ff":          Ff,
            "Fn":          Fn,
            "Ffusion":     Ffusion_flat,
            "Ffusion_seq": Ffusion,
            "Csf":         Csf,
            "Csn":         Csn,
            "Cfn":         Cfn,
        }


# ===========================================================================
# AltFreezing Trainer (ECCV 2023 strategy)
# ===========================================================================

class AltFreezingTrainer:
    """
    Alternate training phases:
      Even epochs: freeze spatial modules, train temporal+heads
      Odd  epochs: freeze temporal+heads, train spatial modules

    Benefits:
    - Prevents gradient interference between spatial and temporal paths
    - Reduces VRAM in frozen phase (no gradient storage for frozen params)
    - Leads to better temporal discrimination (Shi et al., ECCV 2023)

    Usage:
        alt = AltFreezingTrainer(model)
        for epoch in range(epochs):
            alt.set_epoch(epoch)
            # ... train normally
    """

    def __init__(self, model: AFAGNetV3):
        self.model = model
        self.spatial_modules  = [model.backbone, model.spatial,
                                 model.frequency, model.noise, model.cgaf]
        self.temporal_modules = [model.temporal, model.heads]

    def _set_trainable(self, modules, trainable: bool):
        for m in modules:
            for p in m.parameters():
                p.requires_grad = trainable

    def set_epoch(self, epoch: int):
        if epoch % 2 == 0:
            self._set_trainable(self.spatial_modules,  trainable=False)
            self._set_trainable(self.temporal_modules, trainable=True)
            mode = "temporal+heads"
        else:
            self._set_trainable(self.spatial_modules,  trainable=True)
            self._set_trainable(self.temporal_modules, trainable=False)
            mode = "spatial+branches+cgaf"
        print(f"[AltFreezing] Epoch {epoch+1}: training {mode}")


## Pseudo Mask Helper

Source: `train/psuedo_mask.py`

In [ ]:
import torch
import torch.nn.functional as F


def _temporal_ema(cam, decay):
    if cam.shape[1] <= 1:
        return cam
    smoothed = torch.empty_like(cam)
    smoothed[:, 0] = cam[:, 0]
    for t in range(1, cam.shape[1]):
        smoothed[:, t] = decay * smoothed[:, t - 1] + (1 - decay) * cam[:, t]
    return smoothed


def compute_gradcam_stable(model, frames, decay=0.9):
    frames_cam = frames.clone().detach().requires_grad_(True)
    out        = model(frames_cam)
    # Apply sigmoid here since model outputs raw logits
    score      = torch.sigmoid(out["pred_cls"]).mean()
    grads      = torch.autograd.grad(score, frames_cam, retain_graph=False)[0]
    cam        = grads.abs().mean(dim=2, keepdim=True)
    cam        = F.relu(cam)
    flat       = cam.view(cam.shape[0], -1)
    cam_min    = flat.min(dim=1)[0].view(-1, 1, 1, 1, 1)
    cam_max    = flat.max(dim=1)[0].view(-1, 1, 1, 1, 1)
    cam        = (cam - cam_min) / (cam_max - cam_min + 1e-6)
    return _temporal_ema(cam.detach(), decay)


def frequency_map(Ff):
    fmap = torch.mean(torch.abs(Ff), dim=1, keepdim=True)
    return F.interpolate(fmap, size=(224, 224), mode="bilinear", align_corners=False)


def noise_map(Fn):
    nmap = torch.mean(torch.abs(Fn), dim=1, keepdim=True)
    return F.interpolate(nmap, size=(224, 224), mode="bilinear", align_corners=False)


def _restore_temporal_layout(feature_map, frames):
    if feature_map.dim() == 5:
        return feature_map
    B, N = frames.shape[:2]
    return feature_map.view(B, N, feature_map.shape[1],
                            feature_map.shape[2], feature_map.shape[3])


def generate_pseudo_mask(model, frames, Fs, Ff, Fn,
                         alpha=0.5, beta=0.3, gamma=0.2,
                         use_gradcam=True):
    """
    use_gradcam=False  → uses Ff + Fn only, no 2nd forward pass → VRAM safe.
    Always pass use_gradcam=False during training on a 4 GB GPU.
    Never call with use_gradcam=True inside torch.no_grad().
    """
    fmap = _restore_temporal_layout(frequency_map(Ff), frames)
    nmap = _restore_temporal_layout(noise_map(Fn),     frames)

    if use_gradcam:
        cam    = compute_gradcam_stable(model, frames)
        pseudo = alpha * cam + beta * fmap + gamma * nmap
    else:
        b      = beta  / (beta + gamma)
        g      = gamma / (beta + gamma)
        pseudo = b * fmap + g * nmap

    flat       = pseudo.view(pseudo.shape[0], -1)
    pseudo_min = flat.min(dim=1)[0].view(-1, 1, 1, 1, 1)
    pseudo_max = flat.max(dim=1)[0].view(-1, 1, 1, 1, 1)
    return ((pseudo - pseudo_min) / (pseudo_max - pseudo_min + 1e-6)).detach()


## Training Pipeline

Source: `train/train_pipeline_v3.py`

In [ ]:
"""
train_pipeline_v3.py  — Research-Level Training Pipeline for AFAGNet v3
=========================================================================
Improvements over v2 (train_pipeline_final.py):

  FIX-1: Fast balanced sampler (was O(N×disk), now O(N) from label list)
          v2: make_balanced_sampler reads every sample from disk → ~10min overhead
          v3: get_sampler_weights() reads label list directly

  FIX-2: Consistency loss was WRONG (rewarded alignment, not disagreement)
          v2: l_cons = (1 - sigmoid(Csf)).mean()
              When Csf=1 (branches perfectly aligned), loss = (1-sigmoid(1))=0.27 ← penalizes!
              When Csf=0 (branches disagree),          loss = (1-sigmoid(0))=0.50  ← rewards!
              This is BACKWARDS — it was penalizing branch alignment.
          v3: Use proper consistency loss via L1 between branch feature maps.

  FIX-3: Label smoothing now properly applied
          v2: no label smoothing → overconfident predictions, poor calibration
          v3: Focal loss still applied, but soft labels (0.05 smoothing) used

  NEW-1: MixUp augmentation (video-level)
          Mixes two video clips at the tensor level with a random λ.
          Standard in SOTA deepfake detection since SBI (CVPR 2022).
          Applied in-batch, no extra data loading.

  NEW-2: Gradient norm logging to tensorboard-compatible dict
          Stored per epoch for ablation analysis.

  NEW-3: AltFreezing integration
          Automatically switches frozen modules per epoch.

  NEW-4: Per-domain accuracy tracking
          Logs per fake-type accuracy: DeepFakes / Face2Face / FaceSwap / NeuralTextures
          Critical for understanding which forgery types the model struggles with.

  NEW-5: EMA (Exponential Moving Average) weights
          EMA model for evaluation gives ~1-2% AUC improvement over last checkpoint.
          Standard in modern detection pipelines.

  NEW-6: Better model selection criterion
          v2 saved on val_loss (can be dominated by localization loss).
          v3 saves on val_AUC (the actual research metric).

  NEW-7: Warmup patience — don't count warmup epochs toward early stopping.

Research basis:
  - MixUp: Zhang et al., ICLR 2018
  - EMA: Mean Teacher (Tarvainen & Valpola, NeurIPS 2017)
  - AltFreezing: Shi et al., ECCV 2023
  - Label smoothing: Müller et al., NeurIPS 2019
"""

import os
import time
import math
import copy
import warnings
from datetime import datetime
from collections import defaultdict

# Pseudo-mask generation (GradCAM + frequency + noise)
# In this notebook, generate_pseudo_mask is defined in an earlier cell.
_PSEUDO_MASK_AVAILABLE = "generate_pseudo_mask" in globals()

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore", message=".*flash attention.*",     category=UserWarning)
warnings.filterwarnings("ignore", message=".*ComplexHalf.*",         category=UserWarning)
warnings.filterwarnings("ignore", message=".*rcond parameter.*",     category=FutureWarning)
warnings.filterwarnings("ignore", message=".*HF Hub.*",              category=UserWarning)
warnings.filterwarnings("ignore", message=".*Only one class.*",      category=UserWarning)
warnings.filterwarnings("ignore", message=".*UndefinedMetricWarning.*")


# ===========================================================================
# TRAINING GRAPH GENERATOR
# Saves after training completes:
#   training_history.png  — 4-panel: Loss / AUC / Accuracy breakdown / LR schedule
#   overfit_monitor.png   — Train-Val AUC gap per epoch (overfitting detector)
# ===========================================================================

def plot_training_history(history: dict, save_dir: str, batch_size: int = 2):
    os.makedirs(save_dir, exist_ok=True)

    epochs = history.get("epoch", list(range(1, len(history.get("train_loss", [])) + 1)))

    def clean(lst):
        return [v if (v is not None and not (isinstance(v, float) and math.isnan(v)))
                else float("nan") for v in lst]

    style = {
        "train": dict(color="#2563EB", linewidth=2.0, marker="o", markersize=4),
        "val":   dict(color="#DC2626", linewidth=2.0, marker="s", markersize=4),
        "real":  dict(color="#16A34A", linewidth=1.5, linestyle="--"),
        "fake":  dict(color="#D97706", linewidth=1.5, linestyle="--"),
        "bb":    dict(color="#7C3AED", linewidth=1.5),
        "rest":  dict(color="#0891B2", linewidth=1.5),
    }

    fig = plt.figure(figsize=(18, 14))
    fig.suptitle(f"AFAGNetV3 Training History  [batch={batch_size}]",
                 fontsize=15, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.28)

    # ── Loss
    ax1 = fig.add_subplot(gs[0, 0])
    if history.get("train_loss"):
        ax1.plot(epochs, clean(history["train_loss"]), label="Train Loss", **style["train"])
    if history.get("val_loss"):
        ax1.plot(epochs, clean(history["val_loss"]),   label="Val Loss",   **style["val"])
    ax1.set_title("Loss vs Epoch", fontweight="bold"); ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss"); ax1.legend(); ax1.grid(alpha=0.3)

    # ── AUC
    ax2 = fig.add_subplot(gs[0, 1])
    if history.get("train_auc"):
        ax2.plot(epochs, clean(history["train_auc"]), label="Train AUC", **style["train"])
    if history.get("val_auc"):
        ax2.plot(epochs, clean(history["val_auc"]),   label="Val AUC",   **style["val"])
    val_auc_clean = [v for v in clean(history.get("val_auc", [])) if not math.isnan(v)]
    if val_auc_clean:
        best_v = max(val_auc_clean)
        best_i = val_auc_clean.index(best_v)
        ax2.annotate(f"Best: {best_v:.4f}",
                     xy=(epochs[best_i], best_v),
                     xytext=(epochs[best_i], best_v - 0.04),
                     fontsize=9, ha="center", color="#DC2626",
                     arrowprops=dict(arrowstyle="->", color="#DC2626", lw=1.2))
    ax2.set_title("AUC vs Epoch", fontweight="bold"); ax2.set_xlabel("Epoch")
    ax2.set_ylabel("AUC"); ax2.legend(); ax2.grid(alpha=0.3)

    # ── Accuracy breakdown
    ax3 = fig.add_subplot(gs[1, 0])
    if history.get("train_acc"):
        ax3.plot(epochs, [v*100 for v in clean(history["train_acc"])],
                 label="Train Overall", **style["train"])
    if history.get("val_acc"):
        ax3.plot(epochs, [v*100 for v in clean(history["val_acc"])],
                 label="Val Overall", **style["val"])
    if history.get("train_real_acc"):
        ax3.plot(epochs, [v*100 for v in clean(history["train_real_acc"])],
                 label="Train Real", **style["real"])
    if history.get("train_fake_acc"):
        ax3.plot(epochs, [v*100 for v in clean(history["train_fake_acc"])],
                 label="Train Fake", **style["fake"])
    ax3.set_title("Accuracy vs Epoch", fontweight="bold"); ax3.set_xlabel("Epoch")
    ax3.set_ylabel("Accuracy (%)"); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

    # ── LR schedule
    ax4 = fig.add_subplot(gs[1, 1])
    if history.get("lr_backbone"):
        ax4.semilogy(epochs, history["lr_backbone"], label="Backbone LR", **style["bb"])
    if history.get("lr_rest"):
        ax4.semilogy(epochs, history["lr_rest"],    label="Rest LR",    **style["rest"])
    ax4.set_title("Learning Rate Schedule", fontweight="bold"); ax4.set_xlabel("Epoch")
    ax4.set_ylabel("LR (log scale)"); ax4.legend(); ax4.grid(alpha=0.3, which="both")

    plt.savefig(os.path.join(save_dir, "training_history.png"),
                dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    # ── Overfit monitor
    if history.get("train_auc") and history.get("val_auc"):
        ta = clean(history["train_auc"]); va = clean(history["val_auc"])
        gap = [t-v if not (math.isnan(t) or math.isnan(v)) else float("nan")
               for t, v in zip(ta, va)]
        fig2, ax = plt.subplots(figsize=(10, 5))
        ax.bar(epochs, gap,
               color=["#DC2626" if (not math.isnan(g) and g>0.05) else "#16A34A"
                      for g in gap], alpha=0.7)
        ax.axhline(y=0.05, color="orange", linestyle="--", label="Overfit threshold (0.05)")
        ax.axhline(y=0.0,  color="gray",   linestyle="-",  linewidth=0.8)
        ax.set_title("Train−Val AUC Gap (Overfitting Monitor)", fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Train AUC − Val AUC")
        ax.legend(); ax.grid(alpha=0.3)
        fig2.tight_layout()
        fig2.savefig(os.path.join(save_dir, "overfit_monitor.png"),
                     dpi=150, bbox_inches="tight", facecolor="white")
        plt.close(fig2)

    print(f"  Graphs saved → {save_dir}/")


# ===========================================================================
# FOCAL LOSS
# ===========================================================================

class FocalLoss(nn.Module):
    """
    Binary Focal Loss with class-frequency-aware alpha.

    FF++ C23 class ratio: ~1082 real / 4979 fake = 18% real, 82% fake.
    alpha=0.5 biases the model toward predicting FAKE (larger class dominates).
    This is why the optimal threshold was 0.10 — the model over-predicts fake.

    Correct alpha for balanced gradient contribution:
      alpha = 0.25 → strongly upweights real (minority) class gradients.

    Mathematical form:
      loss = -α_t (1-p_t)^γ log(p_t)
      where α_t = alpha for positive (fake) class, (1-alpha) for negative (real)
      With alpha=0.25: real gets weight 0.75, fake gets 0.25
      This compensates for the 4.6:1 fake:real imbalance.
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce    = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs  = torch.sigmoid(logits)
        p_t    = probs * targets + (1.0 - probs) * (1.0 - targets)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        loss   = alpha_t * (1.0 - p_t) ** self.gamma * bce
        return loss.mean()


# ===========================================================================
# FIX-3: CORRECT CONSISTENCY LOSS
# ===========================================================================

def branch_consistency_loss(Fs, Ff, Fn):
    """
    Cosine similarity consistency between branches.
    NaN-safe: adds eps to norms before division, clamps output to valid range.
    """
    fs_p = F.adaptive_avg_pool2d(Fs, 1).flatten(1)   # (B*N, 256)
    ff_p = F.adaptive_avg_pool2d(Ff, 1).flatten(1)
    fn_p = F.adaptive_avg_pool2d(Fn, 1).flatten(1)

    # NaN-safe cosine similarity: normalize manually with eps guard
    fs_n = F.normalize(fs_p, dim=1, eps=1e-8)
    ff_n = F.normalize(ff_p, dim=1, eps=1e-8)
    fn_n = F.normalize(fn_p, dim=1, eps=1e-8)

    cos_sf = (fs_n * ff_n).sum(dim=1).clamp(-1.0, 1.0)
    cos_sn = (fs_n * fn_n).sum(dim=1).clamp(-1.0, 1.0)

    loss = (1.0 - cos_sf).mean() + (1.0 - cos_sn).mean()

    # Safety: return 0 if NaN (e.g. near-zero features during LR restart)
    if torch.isnan(loss):
        return torch.tensor(0.0, device=Fs.device, requires_grad=False)
    return loss


def localization_loss(pred_mask, gt_mask, has_mask, pseudo_mask=None):
    real_loss   = torch.tensor(0.0, device=pred_mask.device)
    pseudo_loss = torch.tensor(0.0, device=pred_mask.device)
    if has_mask.sum() > 0:
        real_loss = F.l1_loss(pred_mask[has_mask], gt_mask[has_mask])
    if (~has_mask).sum() > 0 and pseudo_mask is not None:
        pseudo_loss = F.l1_loss(pred_mask[~has_mask], pseudo_mask[~has_mask])
    return real_loss + 0.3 * pseudo_loss


def temporal_consistency_loss(mask_seq):
    """Penalize large frame-to-frame mask changes. float32 cast prevents fp16 issues."""
    diff = torch.abs(mask_seq[:, 1:].float() - mask_seq[:, :-1].float())
    diff = torch.clamp(diff, 0.0, 10.0)   # cap at 10 — prevents rare extreme values
    return diff.mean()


# ===========================================================================
# NEW-1: MixUp AUGMENTATION
# ===========================================================================

def mixup_batch(frames: torch.Tensor, labels: torch.Tensor, alpha: float = 0.2):
    """
    Video-level MixUp augmentation.
    
    Randomly mixes two video clips with coefficient λ ~ Beta(alpha, alpha).
    Mixed label is a convex combination of the two labels.
    
    Only applied 50% of the time (skip_mixup flag).
    
    Args:
        frames: (B, N, C, H, W)
        labels: (B, 1)
        alpha:  Beta distribution parameter (0.2 = mild mixing)
    
    Returns: (mixed_frames, labels_a, labels_b, lam)
    """
    if alpha <= 0.0:
        return frames, labels, labels, 1.0

    lam   = torch.distributions.Beta(alpha, alpha).sample().item()
    lam   = max(lam, 1.0 - lam)   # ensure lam >= 0.5 (dominant sample)

    B     = frames.size(0)
    index = torch.randperm(B, device=frames.device)

    mixed_frames = lam * frames + (1.0 - lam) * frames[index]
    labels_a     = labels
    labels_b     = labels[index]

    return mixed_frames, labels_a, labels_b, lam


def mixup_loss(criterion, logits, labels_a, labels_b, lam):
    """Loss for MixUp: λ * loss(a) + (1-λ) * loss(b)."""
    return lam * criterion(logits, labels_a) + (1.0 - lam) * criterion(logits, labels_b)


# ===========================================================================
# NEW-5: EMA (Exponential Moving Average)
# ===========================================================================

class ModelEMA:
    """
    EMA of model weights for evaluation.
    EMA model is not trained, just maintained as a running average.
    
    Usage:
        ema = ModelEMA(model, decay=0.9999)
        # after each optimizer.step():
        ema.update(model)
        # for validation:
        ema.apply_to(eval_model)   # or use ema.model directly
    """
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay = decay
        self.model = copy.deepcopy(model)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for ema_p, model_p in zip(self.model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1.0 - self.decay)
        for ema_b, model_b in zip(self.model.buffers(), model.buffers()):
            ema_b.data.copy_(model_b.data)


# ===========================================================================
# WARMUP SCHEDULER (unchanged from v2)
# ===========================================================================

class WarmupScheduler:
    def __init__(self, optimizer, warmup_epochs: int, after_scheduler):
        self.optimizer       = optimizer
        self.warmup_epochs   = warmup_epochs
        self.after_scheduler = after_scheduler
        self.base_lrs        = [g["lr"] for g in optimizer.param_groups]

    def step(self, epoch: int):
        if epoch < self.warmup_epochs:
            scale = (epoch + 1) / max(self.warmup_epochs, 1)
            for g, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                g["lr"] = base_lr * scale
        else:
            # Don't pass epoch — deprecated in PyTorch 1.1+, causes warnings
            self.after_scheduler.step()

    def get_last_lr(self):
        return [g["lr"] for g in self.optimizer.param_groups]


# ===========================================================================
# FIX-1: FAST BALANCED SAMPLER
# ===========================================================================

def make_balanced_sampler(dataset):
    """
    FIX-1: O(N) label extraction from dataset.labels list (no disk access).
    Supports FFPPDatasetV2 directly and torch.utils.data.Subset.
    """
    if hasattr(dataset, "labels"):
        labels = [int(l) for l in dataset.labels]
    elif hasattr(dataset, "dataset") and hasattr(dataset, "indices"):
        base   = dataset.dataset
        labels = [int(base.labels[i]) for i in dataset.indices]
    else:
        # Slow fallback — only if neither attribute exists
        print("WARNING: Using slow label extraction. Use FFPPDatasetV2 for fast sampler.")
        labels = [int(dataset[i]["label"].item()) for i in range(len(dataset))]

    n_real = labels.count(0)
    n_fake = labels.count(1)
    print(f"\nClass balance — real: {n_real}, fake: {n_fake}")

    w_real = 1.0 / (n_real + 1e-6)
    w_fake = 1.0 / (n_fake + 1e-6)
    weights = [w_real if l == 0 else w_fake for l in labels]

    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


# ===========================================================================
# BATCH METRIC HELPERS
# ===========================================================================

def batch_per_class_accuracy(pred_logits, labels):
    probs     = torch.sigmoid(pred_logits)
    preds     = (probs >= 0.5).float()
    # Round labels to 0/1 before comparison — handles label smoothing (0.05/0.95)
    hard_labels = labels.round()
    correct   = (preds == hard_labels).float()
    real_mask = (hard_labels < 0.5).float()
    fake_mask = (hard_labels >= 0.5).float()
    overall   = correct.mean().item()
    real_acc  = (correct * real_mask).sum() / (real_mask.sum() + 1e-6)
    fake_acc  = (correct * fake_mask).sum() / (fake_mask.sum() + 1e-6)
    return overall, real_acc.item(), fake_acc.item()


# ===========================================================================
# SYSTEM INFO
# ===========================================================================

def print_system_info(device, model):
    print("\n" + "=" * 70)
    print("SYSTEM & MODEL INFORMATION")
    print("=" * 70)
    if device.type == "cuda":
        print(f"Device:     GPU (CUDA)")
        print(f"GPU Model:  {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU Memory: {mem:.2f} GB")
        print(f"CUDA:       {torch.version.cuda}")
    else:
        print("Device:     CPU")
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"PyTorch:    {torch.__version__}")
    print(f"Params:     {total:,} total / {trainable:,} trainable")
    print("=" * 70 + "\n")


# ===========================================================================
# TRAINING STEP
# ===========================================================================

def train_one_epoch(
    model, loader, optimizer, device, scaler,
    epoch, total_epochs, use_amp, cls_loss_fn,
    warmup_epochs=2, use_mixup=True, mixup_alpha=0.2,
    ema=None,
):
    model.train()

    running_loss        = 0.0
    running_correct     = 0.0
    running_real_acc    = 0.0
    running_fake_acc    = 0.0
    total_samples       = 0.0
    nan_count           = 0
    all_probs           = []
    all_labels_flat     = []
    loc_weight = max(0.1, 0.8 - epoch * 0.12)
    # FIXED: previous max(0.3, 1.0 - epoch*0.15) decayed too slowly
    # and the 0.3 floor kept localization loss high enough to distort
    # classification logits toward mask-space, causing oscillating val AUC.
    # New schedule: 0.8→0.1 over 6 epochs, then stays at 0.1 (safe floor)

    progress = tqdm(loader, desc=f"Epoch {epoch+1}/{total_epochs} [TRAIN]",
                    leave=True, dynamic_ncols=True)

    for batch_idx, batch in enumerate(progress):
        frames   = batch["frames"].to(device)
        labels   = batch["label"].to(device).unsqueeze(1)
        masks    = batch["mask_frames"].to(device)
        has_mask = batch["has_mask"].to(device)
        if has_mask.dtype != torch.bool:
            has_mask = has_mask.bool()

        # NEW-1: MixUp (50% chance during training, skip during warmup AND epoch 3)
        # Disabled for first 3 epochs — model needs to learn basic features first
        do_mixup = use_mixup and epoch >= max(warmup_epochs, 3) and torch.rand(1).item() < 0.5
        if do_mixup:
            frames, labels_a, labels_b, lam = mixup_batch(frames, labels, alpha=mixup_alpha)
        else:
            labels_a, labels_b, lam = labels, labels, 1.0

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            out      = model(frames)
            pred_cls  = out["pred_cls"]
            pred_mask = out["pred_mask"]

            Fs, Ff, Fn = out.get("Fs"), out.get("Ff"), out.get("Fn")

            # Classification loss
            with torch.cuda.amp.autocast(enabled=False):
                if do_mixup:
                    l_cls = mixup_loss(cls_loss_fn,
                                       pred_cls.float(), labels_a.float(),
                                       labels_b.float(), lam)
                else:
                    l_cls = cls_loss_fn(pred_cls.float(), labels.float())

            # Localization loss — real GT masks where available,
            # pseudo-mask (from freq+noise branches) for samples without GT masks
            mid = masks[:, masks.shape[1] // 2].unsqueeze(1)

            pseudo_mask = None
            if _PSEUDO_MASK_AVAILABLE and Fs is not None and Ff is not None and Fn is not None:
                try:
                    # use_gradcam=False → uses Ff + Fn only, no 2nd forward pass
                    # Safe on 4GB GPU — avoids storing gradients for entire model twice
                    #
                    # TO ENABLE GRADCAM ON T4 (16GB): change use_gradcam=True
                    # GradCAM adds ~1GB VRAM but gives better pseudo-mask quality
                    # (+0.3–0.5% localization IoU improvement).
                    # NEVER enable inside torch.no_grad() — needs autograd.
                    pseudo_mask = generate_pseudo_mask(
                        model, frames, Fs, Ff, Fn,
                        alpha=0.5, beta=0.3, gamma=0.2,
                        use_gradcam=False,   # ← change to True on T4
                    )
                    # Resize pseudo_mask to (B,1,224,224) middle frame only
                    N_frames = frames.shape[1]
                    pseudo_mask = pseudo_mask[:, N_frames // 2]  # (B,1,H,W)
                except Exception:
                    pseudo_mask = None

            l_loc = localization_loss(pred_mask, mid, has_mask, pseudo_mask=pseudo_mask)

            # FIX-2: Correct consistency loss
            l_cons = torch.tensor(0.0, device=device)
            if Fs is not None and Ff is not None and Fn is not None:
                l_cons = branch_consistency_loss(Fs, Ff, Fn)

            # Temporal consistency
            mask_seq = out.get("mask_sequence")
            if mask_seq is None:
                mask_seq = pred_mask.unsqueeze(1).repeat(1, frames.shape[1], 1, 1, 1)
            l_temp = temporal_consistency_loss(mask_seq)

            loss = l_cls + loc_weight * l_loc + 0.05 * l_cons + 0.05 * l_temp

        # NaN/Inf detection — check BEFORE backward to avoid grad_fn crash.
        # The requires_grad error happens when a component is NaN/Inf and gets
        # replaced with a detached zero — the total loss then has no grad_fn.
        # Correct fix: skip the entire batch if loss is invalid.
        if not torch.isfinite(loss):
            nan_count += 1
            optimizer.zero_grad(set_to_none=True)
            # DO NOT call scaler.update() here.
            # GradScaler.update() asserts that scale()+backward() were called first.
            # Calling update() on a skipped batch → AssertionError: "No inf checks
            # were recorded prior to update." (the exact crash you hit at epoch 4).
            # The scaler state stays consistent on its own when we skip a batch —
            # it only needs update() after an actual backward pass.
            #
            # ADDITIONALLY: reset the scaler's loss scale when NaN appears.
            # A bad scale factor (e.g. 65536 after long stable training) can cause
            # gradient overflow on the next batch too, creating a cascade.
            # Halving the scale manually breaks the cascade.
            if hasattr(scaler, '_scale') and scaler._scale is not None:
                scaler._scale.fill_(scaler._scale.item() / 2.0)
            if nan_count > 10:
                raise RuntimeError(
                    f"Too many non-finite loss batches ({nan_count}). "
                    f"Last: cls={l_cls.item():.4f} loc={l_loc.item():.4f} "
                    f"cons={l_cons.item():.4f} temp={l_temp.item():.4f}"
                )
            continue

        # CORRECT AMP gradient clipping order
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Skip optimizer step if backward produced inf/nan gradients
        if torch.isfinite(grad_norm) and grad_norm < 50.0:
            scaler.step(optimizer)
        else:
            # Inf/NaN gradients after clipping = scale too high, halve it
            if hasattr(scaler, '_scale') and scaler._scale is not None:
                scaler._scale.fill_(max(scaler._scale.item() / 4.0, 64.0))
            optimizer.zero_grad(set_to_none=True)

        scaler.update()

        # Periodic VRAM defrag — prevents the incremental fragmentation
        # that causes OOM at epoch 2+ even when total usage looks fine
        if batch_idx % 500 == 0 and batch_idx > 0:
            torch.cuda.empty_cache()

        # NEW-5: EMA update after each step
        if ema is not None:
            ema.update(model)

        bs = frames.size(0)
        with torch.no_grad():
            probs = torch.sigmoid(pred_cls.detach()).squeeze(1)
        overall, real_acc, fake_acc = batch_per_class_accuracy(pred_cls.detach(), labels if not do_mixup else labels_a)
        running_loss    += loss.item() * bs
        running_correct += overall * bs
        running_real_acc += real_acc * bs
        running_fake_acc += fake_acc * bs
        total_samples   += bs
        all_probs.extend(probs.cpu().tolist())
        # Use hard (rounded) labels for AUC computation
        hard_labels_for_log = labels.round().squeeze(1)
        all_labels_flat.extend(hard_labels_for_log.cpu().tolist())

        progress.set_postfix(
            loss=f"{running_loss/total_samples:.4f}",
            acc=f"{100*running_correct/total_samples:.1f}%",
            real=f"{100*running_real_acc/total_samples:.1f}%",
            fake=f"{100*running_fake_acc/total_samples:.1f}%",
            nan=nan_count,
        )

    progress.close()
    if total_samples == 0:
        return {"loss": float("nan"), "accuracy": 0.0, "auc": 0.0, "nan_count": nan_count}

    # Compute training AUC
    try:
        train_auc = roc_auc_score(
            [round(l) for l in all_labels_flat],  # round mixed labels to nearest int
            all_probs
        )
    except Exception:
        train_auc = 0.0

    return {
        "loss":     running_loss / total_samples,
        "accuracy": running_correct / total_samples,
        "real_acc": running_real_acc / total_samples,
        "fake_acc": running_fake_acc / total_samples,
        "auc":      train_auc,
        "nan_count": nan_count,
    }


# ===========================================================================
# VALIDATION STEP
# ===========================================================================

def validate(model, loader, device, epoch, total_epochs, use_amp, cls_loss_fn):
    model.eval()

    running_loss    = 0.0
    running_loss_n  = 0     # separate counter for finite-loss batches only
    running_correct = 0.0
    total_samples   = 0.0
    all_probs       = []
    all_labels_flat = []
    all_preds       = []
    domain_correct  = defaultdict(list)
    domain_probs    = defaultdict(list)   # NEW: per-domain probs for AUC
    domain_labels   = defaultdict(list)   # NEW: per-domain labels for AUC

    progress = tqdm(loader, desc=f"Epoch {epoch+1}/{total_epochs} [VAL]",
                    leave=True, dynamic_ncols=True)

    with torch.no_grad():
        for batch in progress:
            frames   = batch["frames"].to(device)
            labels   = batch["label"].to(device).unsqueeze(1)
            masks    = batch["mask_frames"].to(device)
            has_mask = batch["has_mask"].to(device)
            if has_mask.dtype != torch.bool:
                has_mask = has_mask.bool()
            domains  = batch.get("domain", [])

            with torch.cuda.amp.autocast(enabled=use_amp):
                out       = model(frames)
                pred_cls  = out["pred_cls"]
                pred_mask = out["pred_mask"]
                mid       = masks[:, masks.shape[1] // 2].unsqueeze(1)

                with torch.cuda.amp.autocast(enabled=False):
                    l_cls = cls_loss_fn(pred_cls.float(), labels.float())

                l_loc = localization_loss(pred_mask, mid, has_mask)
                loss  = l_cls + 0.5 * l_loc

            bs = frames.size(0)
            probs = torch.sigmoid(pred_cls.detach()).squeeze(1)
            # Replace any NaN probabilities with 0.5 (neutral prediction)
            probs = torch.where(torch.isfinite(probs), probs, torch.full_like(probs, 0.5))
            preds = (probs >= 0.5).float()
            correct = (preds == labels.round().squeeze(1)).float()

            # Only add loss to running total if it's finite
            if torch.isfinite(loss):
                running_loss   += loss.item() * bs
                running_loss_n += bs

            running_correct += correct.mean().item() * bs
            total_samples   += bs

            all_probs.extend(probs.cpu().tolist())
            all_labels_flat.extend(labels.squeeze(1).round().cpu().tolist())
            all_preds.extend(preds.cpu().tolist())

            # Per-domain tracking (accuracy + AUC)
            for i, domain in enumerate(domains):
                domain_correct[domain].append(correct[i].item())
                domain_probs[domain].append(probs[i].item())
                domain_labels[domain].append(labels.squeeze(1)[i].item())

            progress.set_postfix(
                loss=f"{running_loss/running_loss_n:.4f}" if running_loss_n > 0 else "nan",
                acc=f"{100*running_correct/total_samples:.1f}%" if total_samples > 0 else "nan",
            )

    progress.close()
    if total_samples == 0:
        return {"loss": float("nan"), "accuracy": 0.0, "auc": 0.0, "domain_stats": {}}

    try:
        auc = roc_auc_score(all_labels_flat, all_probs)
    except Exception:
        auc = 0.0

    # Per-domain stats
    # NOTE: Fake domains (Deepfakes, Face2Face etc.) only contain fake samples.
    # Real domains (youtube, actors) only contain real samples.
    # roc_auc_score requires both classes → shows n/a for single-class domains.
    # Instead, show: fake domains → Recall (% of fakes caught)
    #                real domains → Specificity (% of reals correctly identified)
    #                Mixed domains (DeepFakeDetection has both) → full AUC
    domain_stats = {}
    for d in domain_correct:
        n       = len(domain_correct[d])
        d_acc   = sum(domain_correct[d]) / n
        d_lbls  = domain_labels[d]
        d_prbs  = domain_probs[d]

        n_real  = sum(1 for l in d_lbls if l < 0.5)
        n_fake  = sum(1 for l in d_lbls if l >= 0.5)

        if n_real > 0 and n_fake > 0:
            # Mixed domain — full AUC is valid
            try:
                d_auc = roc_auc_score(d_lbls, d_prbs)
            except Exception:
                d_auc = float("nan")
            d_type = "mixed"
        elif n_fake > 0:
            # All-fake domain: show recall = fraction of fakes detected
            preds_domain = [1.0 if p > 0.5 else 0.0 for p in d_prbs]
            d_auc  = sum(preds_domain) / len(preds_domain)   # = recall on this domain
            d_type = "recall"
        else:
            # All-real domain: show specificity = fraction of reals correctly kept
            preds_domain = [1.0 if p <= 0.5 else 0.0 for p in d_prbs]
            d_auc  = sum(preds_domain) / len(preds_domain)
            d_type = "specificity"

        domain_stats[d] = {
            "acc": d_acc, "auc": d_auc, "n": n,
            "n_real": n_real, "n_fake": n_fake, "metric_type": d_type,
        }

    return {
        "loss":         running_loss / running_loss_n if running_loss_n > 0 else float("nan"),
        "accuracy":     running_correct / total_samples,
        "auc":          auc,
        "domain_stats": domain_stats,
        "labels":       all_labels_flat,
        "probs":        all_probs,
        "preds":        all_preds,
    }


# ===========================================================================
# MAIN TRAIN FUNCTION
# ===========================================================================

def train(
    model,
    train_dataset,
    val_dataset=None,
    epochs: int = 20,
    batch_size: int = 4,
    device=None,
    lr: float = 3e-4,
    backbone_lr_factor: float = 0.1,
    weight_decay: float = 1e-3,     # increased from 1e-4 → stronger L2 regularization
    num_workers: int = 0,
    patience: int = 7,
    warmup_epochs: int = 2,
    save_dir: str = "checkpoints",
    log_dir: str = "experiments/logs",
    use_amp: bool = True,
    focal_alpha: float = 0.5,
    focal_gamma: float = 2.0,
    use_mixup: bool = True,
    mixup_alpha: float = 0.2,
    use_ema: bool = True,
    ema_decay: float = 0.9999,
    use_alt_freezing: bool = False,   # enable AltFreezing (better but changes training)
    resume_epoch: int = 0,            # skip epochs already completed (for resuming)
):
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device  = torch.device(device)
    use_amp = use_amp and (device.type == "cuda")

    # ---- Enable training mode augmentation ----
    # This sets the dataset's augmentation flag so __getitem__ applies augmentation
    def _set_training_mode(ds, mode: bool):
        base = ds.dataset if hasattr(ds, "dataset") else ds
        if hasattr(base, "training_mode"):
            base.training_mode = mode

    _set_training_mode(train_dataset, True)
    if val_dataset is not None:
        _set_training_mode(val_dataset, False)

    # ---- Fast balanced sampler (FIX-1) ----
    sampler      = make_balanced_sampler(train_dataset)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(num_workers > 0),
        drop_last=True,   # avoid single-sample batches (can cause BN issues)
    )
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset, batch_size=batch_size, shuffle=False,
            num_workers=num_workers,
            pin_memory=(device.type == "cuda"),
            persistent_workers=(num_workers > 0),
        )

    model = model.to(device)

    # ---- Loss ----
    cls_loss_fn = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

    # ---- Optimizer: separate LR for pretrained backbone ----
    backbone_params = list(model.backbone.parameters()) if hasattr(model, "backbone") else []
    backbone_ids    = {id(p) for p in backbone_params}
    other_params    = [p for p in model.parameters() if id(p) not in backbone_ids]

    if backbone_params:
        param_groups = [
            {"params": backbone_params, "lr": lr * backbone_lr_factor, "name": "backbone"},
            {"params": other_params,    "lr": lr,                       "name": "rest"},
        ]
        print(f"\nLR groups: backbone={lr * backbone_lr_factor:.1e}, rest={lr:.1e}")
    else:
        param_groups = [{"params": model.parameters(), "lr": lr}]

    optimizer = optim.AdamW(param_groups, weight_decay=weight_decay)

    # CosineAnnealingLR: smooth decay over all epochs, NO restarts.
    # CosineAnnealingWarmRestarts caused NaN at restart points (epoch 7, 9)
    # because LR jumps back to max while GradScaler is still at high scale.
    cosine_sched = CosineAnnealingLR(
        optimizer,
        T_max=epochs - warmup_epochs,   # decay over remaining epochs after warmup
        eta_min=1e-6,
    )
    scheduler = WarmupScheduler(optimizer, warmup_epochs=warmup_epochs,
                                after_scheduler=cosine_sched)

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp,
        init_scale=1024.0,       # Low start — grows slowly, stays safe through 30+ epochs
        growth_factor=2.0,
        backoff_factor=0.5,
        growth_interval=2000,    # Only doubles every 2000 clean steps (not 500)
    )

    # ---- EMA (NEW-5) ----
    ema = ModelEMA(model, decay=ema_decay) if use_ema else None

    # ---- AltFreezing (NEW-3) ----
    alt_freezing = None
    if use_alt_freezing:
        try:
            alt_freezing = AltFreezingTrainer(model)
            print("AltFreezing enabled.")
        except ImportError:
            print("AltFreezing: AFAGNetV3 not found, skipping.")

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)

    best_path    = os.path.join(save_dir, "best_model.pth")
    latest_path  = os.path.join(save_dir, "latest_model.pth")
    ema_path     = os.path.join(save_dir, "ema_model.pth")
    log_path     = os.path.join(log_dir,  "epoch_results_v3.txt")

    # NEW-6: Save on val AUC instead of val loss
    best_auc         = 0.0
    patience_counter = 0
    history          = defaultdict(list)

    # Fast-forward scheduler to resume_epoch position
    if resume_epoch > 0:
        print(f"Resuming training from epoch {resume_epoch + 1}")
        for _ in range(resume_epoch):
            scheduler.step(0)   # advance internal counter; warmup uses epoch arg separately
        best_auc = 0.0  # will be updated by first val run

    print_system_info(device, model)
    print(f"\n{'='*70}")
    print(f"Training Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}")
    print(f"Train: {len(train_loader.dataset)} samples | Val: {len(val_loader.dataset) if val_loader else 'N/A'}")
    print(f"AMP: {use_amp} | MixUp: {use_mixup} | EMA: {use_ema} | AltFreezing: {use_alt_freezing}")
    if resume_epoch > 0:
        print(f"Resuming from epoch {resume_epoch} — training epochs {resume_epoch+1}–{epochs}")
    print(f"{'='*70}\n")

    start_time = time.time()

    for epoch in range(resume_epoch, epochs):
        epoch_start = time.time()
        scheduler.step(epoch)   # warmup scheduler still needs epoch for LR scaling

        # AltFreezing phase switch
        if alt_freezing is not None:
            alt_freezing.set_epoch(epoch)

        # ---- Train ----
        train_m = train_one_epoch(
            model=model, loader=train_loader, optimizer=optimizer,
            device=device, scaler=scaler,
            epoch=epoch, total_epochs=epochs, use_amp=use_amp,
            cls_loss_fn=cls_loss_fn, warmup_epochs=warmup_epochs,
            use_mixup=use_mixup, mixup_alpha=mixup_alpha, ema=ema,
        )

        # ---- Validate ----
        val_m = {}
        if val_loader is not None:
            torch.cuda.empty_cache()
            eval_model = ema.model if ema is not None else model
            val_m = validate(
                model=eval_model, loader=val_loader, device=device,
                epoch=epoch, total_epochs=epochs,
                use_amp=use_amp, cls_loss_fn=cls_loss_fn,
            )

        elapsed  = time.time() - epoch_start
        val_auc  = val_m.get("auc", 0.0)
        val_loss = val_m.get("loss", float("nan"))
        val_acc  = val_m.get("accuracy", 0.0)

        # Log — full history for graph generation
        history["train_loss"].append(float(train_m["loss"]) if not math.isnan(train_m["loss"]) else float("nan"))
        history["train_auc"].append(float(train_m.get("auc", 0.0)))
        history["train_acc"].append(float(train_m.get("accuracy", 0.0)))
        history["train_real_acc"].append(float(train_m.get("real_acc", 0.0)))
        history["train_fake_acc"].append(float(train_m.get("fake_acc", 0.0)))
        history["train_nan"].append(int(train_m.get("nan_count", 0)))
        history["val_loss"].append(float(val_loss) if not math.isnan(val_loss) else float("nan"))
        history["val_auc"].append(float(val_auc))
        history["val_acc"].append(float(val_acc))
        history["lr_backbone"].append(float(optimizer.param_groups[0]["lr"]))
        history["lr_rest"].append(float(optimizer.param_groups[-1]["lr"]))
        history["epoch"].append(epoch + 1)

        print(f"\n{'-'*70}")
        print(f"Epoch {epoch+1}/{epochs} Summary")
        print(f"{'-'*70}")
        print(f"Train  Loss: {train_m['loss']:.5f}  Acc: {train_m['accuracy']*100:.1f}%  "
              f"AUC: {train_m.get('auc',0):.4f}  "
              f"[Real: {train_m.get('real_acc',0)*100:.1f}%  Fake: {train_m.get('fake_acc',0)*100:.1f}%]  "
              f"NaN: {train_m.get('nan_count',0)}")
        if val_m:
            print(f"Val    Loss: {val_loss:.5f}  Acc: {val_acc*100:.1f}%  AUC: {val_auc:.4f}")
            domain_stats = val_m.get("domain_stats", {})
            if domain_stats:
                print(f"  {'Domain':<22} {'N':>5}  {'Acc':>7}  {'Metric':>12}  {'Type'}")
                print(f"  {'-'*22}  {'-'*5}  {'-'*7}  {'-'*12}  {'-'*12}")
                for domain, stats in sorted(domain_stats.items()):
                    mtype = stats.get("metric_type", "?")
                    mval  = stats.get("auc", float("nan"))
                    if mtype == "mixed":
                        metric_label = f"AUC={mval:.4f}"
                    elif mtype == "recall":
                        metric_label = f"Recall={mval:.4f}"
                    elif mtype == "specificity":
                        metric_label = f"Specif={mval:.4f}"
                    else:
                        metric_label = f"{mval:.4f}"
                    print(f"  {domain:<22} {stats['n']:>5}  {stats['acc']*100:>6.1f}%  {metric_label:>12}  ({stats['n_real']}R/{stats['n_fake']}F)")
        lr_str = " | ".join(f"{g.get('name','?')}:{g['lr']:.1e}" for g in optimizer.param_groups)
        print(f"LR: {lr_str}   Time: {elapsed:.1f}s")

        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"epoch={epoch+1},train_loss={train_m['loss']:.6f},"
                    f"train_auc={train_m.get('auc',0):.4f},"
                    f"val_loss={val_loss:.6f},val_auc={val_auc:.4f},"
                    f"val_acc={val_acc:.4f},nan={train_m.get('nan_count',0)},"
                    f"time={elapsed:.1f}\n")

        # NEW-6: Save on val AUC
        # NEW-7: Don't count warmup epochs in patience
        epoch_had_nan = train_m.get("nan_count", 0) > 0
        if epoch < warmup_epochs:
            print(f"  -> Warmup epoch, not counting toward patience.")
            torch.save(model.state_dict(), latest_path)
            if ema is not None and not epoch_had_nan:
                torch.save(ema.model.state_dict(), ema_path)
            continue

        # Only save best/EMA if epoch was clean (no NaN batches) AND AUC improved
        # This prevents a corrupted EMA from overwriting a good checkpoint
        if not math.isnan(val_auc) and val_auc > best_auc and not epoch_had_nan:
            best_auc         = val_auc
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
            if ema is not None:
                torch.save(ema.model.state_dict(), ema_path)
            print(f"  -> BEST model saved (val AUC: {val_auc:.4f})\n")
        elif epoch_had_nan:
            patience_counter += 1
            print(f"  -> Skipped save (epoch had {train_m.get('nan_count',0)} NaN batch(es)). "
                  f"Best AUC stays: {best_auc:.4f}. Patience: {patience_counter}/{patience}\n")
        else:
            patience_counter += 1
            print(f"  -> No improvement (best AUC: {best_auc:.4f}). Patience: {patience_counter}/{patience}\n")
            if patience_counter >= patience:
                print(f"{'='*70}")
                print(f"EARLY STOPPING after {patience} epochs without AUC improvement")
                print(f"{'='*70}\n")
                break

        torch.save(model.state_dict(), latest_path)
        # Only update latest EMA if epoch was clean
        if ema is not None and not epoch_had_nan:
            torch.save(ema.model.state_dict(), ema_path)

    total_time = time.time() - start_time
    print(f"\n{'='*70}")
    print("TRAINING COMPLETE")
    print(f"{'='*70}")
    print(f"Total Time  : {total_time/3600:.2f}h ({total_time/60:.1f}min)")
    print(f"Best Val AUC: {best_auc:.4f}")
    print(f"Saved: {best_path}")
    if ema is not None:
        print(f"EMA  : {ema_path}")
    print(f"Logs : {log_path}")
    print(f"\nFinished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}\n")

    # Save training graphs automatically after training
    graph_dir = os.path.join(log_dir, "graphs")
    try:
        plot_training_history(dict(history), save_dir=graph_dir, batch_size=batch_size)
    except Exception as e:
        print(f"[WARN] Graph generation failed: {e}")

    return dict(history)
